# Multi-head attention, step by step

Keep the river-bank example from Part II. First follow two different reading
patterns through the lecture's figures. Then run the detailed implementation
lab, including the batch axes and a numerical comparison with PyTorch.

[Visual story](#visual-story) · [Executable lab](#executable-lab)

The toy uses hand-chosen parameters. The final links lead to genuinely trained
TinyStories models, not this worksheet. As in Part II, E stores embedding rows,
M is the mask, H = AV stores message rows, and E′ = E + ΔE. Superscripts label
heads; subscripts label tokens.

[Part III](../../part3.html) · [Live models](../../word-lab/) · [Download code and notebooks](wordlm-notebooks.zip)

Run all cells from this directory. No data download or training run is needed.
The single optimizer step demonstrates learning; it does not create the models
used in the benchmark.

In [1]:
import json
import math
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display, HTML
from multihead_from_scratch import (TinyMultiHeadLM, ScratchMultiHead,
                                   load_worksheet_weights, copy_to_pytorch)

torch.manual_seed(7)
worksheet = json.loads(Path('multihead-worksheet.json').read_text())
word_to_id = {word: i for i, word in enumerate(worksheet['vocab'])}
river_ids = [word_to_id[w.lower()] for w in worksheet['sentences']['river']]
cheque_ids = [word_to_id[w.lower()] for w in worksheet['sentences']['cheque']]
print('River tokens:', worksheet['sentences']['river'])
print('River IDs:', river_ids)

River tokens: ['The', 'fisherman', 'sat', 'beside', 'the', 'river', 'bank', 'and', 'watched', 'the']
River IDs: [0, 1, 2, 3, 0, 4, 5, 6, 7, 0]


<a id="visual-story"></a>
# The visual story

The same figures appear in the lecture. Short code excerpts here are explained visually; the executable lab below builds their inputs and runs each operation.

<a id="s01-v-prefix"></a>
## Back to the river bank

[Matching slide](../../part3.html?present#s01/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 345" role="img" aria-label="The river-bank prefix from Part II" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The river-bank prefix from Part II</title><rect x="20" y="90" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="90" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="90" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="90" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="90" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="90" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="90" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="90" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="90" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="121" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="90" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="121" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="78" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="122" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><text x="560" y="238" font-size="32" fill="#14171f" text-anchor="middle" font-weight="500">The known prefix ends here.</text><text x="560" y="288" font-size="28" fill="#586174" text-anchor="middle" font-weight="500">The updated final “the” will predict the next word.</text></svg>

Which parts of this prefix would help you choose a continuation?



<a id="s01-v-shared"></a>
## One head mixes the value rows

[Matching slide](../../part3.html?present#s01/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 435" role="img" aria-label="One weight per source multiplies every value coordinate" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>One weight per source multiplies every value coordinate</title><text x="24" y="32" font-size="23" fill="#586174" text-anchor="start" font-weight="500">Two-source illustration: invented values and weights</text><text x="24" y="91" font-size="25" fill="#586174" text-anchor="start" font-weight="600">Source</text><text x="350" y="91" font-size="25" fill="#586174" text-anchor="start" font-weight="600">Setting feature</text><text x="675" y="91" font-size="25" fill="#586174" text-anchor="start" font-weight="600">Person feature</text><text x="24" y="140" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">river</text><text x="390" y="140" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">10</text><text x="715" y="140" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">1</text><text x="24" y="192" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">fisherman</text><text x="390" y="192" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">2</text><text x="715" y="192" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">8</text><path d="M24 215 H1120" fill="none" stroke="#d9dfe9" stroke-width="2"/><text x="24" y="267" font-size="27" fill="#be123c" text-anchor="start" font-weight="600">One head: river gets 0.8, fisherman gets 0.2</text><text x="580" y="327" font-size="33" fill="#0f766e" text-anchor="middle" font-weight="500">0.8 × [10, 1] + 0.2 × [2, 8] = [8.4, 2.4]</text><text x="580" y="399" font-size="27" fill="#14171f" text-anchor="middle" font-weight="500">Both output coordinates use the same 80% / 20% mixture.</text></svg>

Suppose we want setting clues from river and person clues from fisherman. A single head uses one source weight for every coordinate of each value.

<p>This two-source illustration uses invented values and normalized weights, separately from the ten-token worksheet that follows. A weight of 0.8 on river scales both of its value coordinates. A head can attend to multiple sources, but cannot choose a separate attention weight for each value coordinate.</p>

<a id="s01-v-independent"></a>
## Two heads can choose different mixtures

[Matching slide](../../part3.html?present#s01/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 435" role="img" aria-label="Two heads weight the sources independently" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Two heads weight the sources independently</title><text x="24" y="32" font-size="23" fill="#586174" text-anchor="start" font-weight="500">Two-source illustration: invented values and weights</text><text x="24" y="91" font-size="25" fill="#586174" text-anchor="start" font-weight="600">Source</text><text x="350" y="91" font-size="25" fill="#586174" text-anchor="start" font-weight="600">Setting feature</text><text x="675" y="91" font-size="25" fill="#586174" text-anchor="start" font-weight="600">Person feature</text><text x="24" y="140" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">river</text><text x="390" y="140" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">10</text><text x="715" y="140" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">1</text><text x="24" y="192" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">fisherman</text><text x="390" y="192" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">2</text><text x="715" y="192" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">8</text><path d="M24 215 H1120" fill="none" stroke="#d9dfe9" stroke-width="2"/><text x="24" y="273" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 1: setting</text><text x="390" y="273" font-size="30" fill="#0f766e" text-anchor="start" font-weight="500">0.8 × 10 + 0.2 × 2 = 8.4</text><text x="24" y="333" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 2: person</text><text x="390" y="333" font-size="30" fill="#0f766e" text-anchor="start" font-weight="500">0.2 × 1 + 0.8 × 8 = 6.6</text><text x="580" y="399" font-size="26" fill="#be123c" text-anchor="middle" font-weight="500">river gets 0.8 in one head; fisherman gets 0.8 in the other.</text></svg>

Let one head return the setting feature and another return the person feature. Each head can choose its own source weights. Their outputs stay separate until the output projection.

<p>Both heads receive both source rows. For this illustration, one value projection selects the setting coordinate and the other selects the person coordinate. The total output width stays two. The example demonstrates independent weighting, not a guarantee that two trained heads outperform every one-head model. The following slides return to our ten-token, four-coordinate worksheet.</p>

In [2]:
# Separate two-source illustration, before the ten-token worksheet.
values = torch.tensor([[10., 1.], [2., 8.]])  # river, fisherman
setting_weights = torch.tensor([0.8, 0.2])
person_weights = torch.tensor([0.2, 0.8])
one_head = setting_weights @ values
two_heads = torch.stack([setting_weights @ values[:, 0],
                         person_weights @ values[:, 1]])
print('One shared mixture:', one_head.tolist())
print('Two separate mixtures:', two_heads.tolist())
torch.testing.assert_close(one_head, torch.tensor([8.4, 2.4]))
torch.testing.assert_close(two_heads, torch.tensor([8.4, 6.6]))

One shared mixture: [8.399999618530273, 2.4000000953674316]
Two separate mixtures: [8.399999618530273, 6.599999904632568]


<a id="s01-v-one"></a>
## Setting clues in the full sentence

[Matching slide](../../part3.html?present#s01/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Head 1 reads the known prefix" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Head 1 reads the known prefix</title><text x="20" y="32" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 1: setting clues</text><rect x="20" y="69" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="69" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="69" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="69" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="69" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="69" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="69" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="69" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="69" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="69" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="100" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="101" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><path d="M54.5 124 L830 208 M820.4235972807309 210.8796720227094 L830 208 L821.2623057087912 203.1365959993665" fill="none" stroke="#be123c" stroke-width="0.6077888739354684"/><path d="M170.5 124 L830 208 M820.3711805086315 210.69922863104298 L830 208 L821.3552287580618 202.97327838700437" fill="none" stroke="#be123c" stroke-width="2.04441104465339"/><path d="M284.5 124 L830 208 M820.3040180798132 210.4470256646408 L830 208 L821.4893555773796 202.74938751080228" fill="none" stroke="#be123c" stroke-width="0.6300031000084825"/><path d="M380.5 124 L830 208 M820.2307837513852 210.1359807788928 L830 208 L821.6614626727436 202.48014536043274" fill="none" stroke="#be123c" stroke-width="0.6607565449036875"/><path d="M476.0 124 L830 208 M820.1391557312623 209.66245309939978 L830 208 L821.9373127814537 202.08450553073624" fill="none" stroke="#be123c" stroke-width="0.6179947723052064"/><path d="M565.0 124 L830 208 M820.0432490444437 208.92903735609943 L830 208 L822.396614199894 201.50473061569065" fill="none" stroke="#be123c" stroke-width="7.008797895659618"/><path d="M664.0 124 L830 208 M820.0234214675627 207.31598188173805 L830 208 L823.5399353635429 200.36668061111052" fill="none" stroke="#be123c" stroke-width="1.0286863304536027"/><path d="M753.0 124 L830 208 M820.9055505868265 203.84175639587716 L830 208 L826.6467774528851 198.5789651019901" fill="none" stroke="#be123c" stroke-width="0.5991148633565428"/><path d="M859.5 124 L830 208 M829.3777520839247 198.0193783995715 L830 208 L836.7261353980238 200.6000606348801" fill="none" stroke="#be123c" stroke-width="0.7033317113674585"/><path d="M967.0 124 L830 208 M835.816633008165 199.86570344477622 L830 208 L839.8876744216715 206.50537813109042" fill="none" stroke="#be123c" stroke-width="0.5991148633565428"/><text x="565.0" y="152" font-size="22" fill="#be123c" text-anchor="middle" font-weight="650">0.718</text><rect x="747" y="214" width="165" height="45" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="830" y="244" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">10 · the</text><text x="22" y="230" font-size="24" fill="#586174" text-anchor="start" font-weight="500">Thicker arrow = more attention weight</text><text x="946" y="243" font-size="22" fill="#586174" text-anchor="start" font-weight="500">receiver</text></svg>

Back to all ten tokens. Our hand-chosen head 1 gives river a large weight. Its values carry setting information to the receiver.

<p>These are hand-chosen two-head parameters applied to Part II’s exact input rows. “Setting” and “person” name the intended behaviour of this example, not jobs assigned to trained heads. Arrows show information moving from source to receiver; their widths encode computed attention weights.</p>

<a id="s01-v-two"></a>
## Another head can read who is there

[Matching slide](../../part3.html?present#s01/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Head 2 reads the known prefix" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Head 2 reads the known prefix</title><text x="20" y="32" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 2: person clues</text><rect x="20" y="69" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="69" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="69" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="69" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="69" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="69" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="69" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="69" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="69" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="69" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="100" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="101" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><path d="M54.5 124 L830 208 M820.4235972807309 210.8796720227094 L830 208 L821.2623057087912 203.1365959993665" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M170.5 124 L830 208 M820.3711805086315 210.69922863104298 L830 208 L821.3552287580618 202.97327838700437" fill="none" stroke="#be123c" stroke-width="6.786645910206812"/><path d="M284.5 124 L830 208 M820.3040180798132 210.4470256646408 L830 208 L821.4893555773796 202.74938751080228" fill="none" stroke="#be123c" stroke-width="1.0938532271216301"/><path d="M380.5 124 L830 208 M820.2307837513852 210.1359807788928 L830 208 L821.6614626727436 202.48014536043274" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M476.0 124 L830 208 M820.1391557312623 209.66245309939978 L830 208 L821.9373127814537 202.08450553073624" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><path d="M565.0 124 L830 208 M820.0432490444437 208.92903735609943 L830 208 L822.396614199894 201.50473061569065" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><path d="M664.0 124 L830 208 M820.0234214675627 207.31598188173805 L830 208 L823.5399353635429 200.36668061111052" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M753.0 124 L830 208 M820.9055505868265 203.84175639587716 L830 208 L826.6467774528851 198.5789651019901" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M859.5 124 L830 208 M829.3777520839247 198.0193783995715 L830 208 L836.7261353980238 200.6000606348801" fill="none" stroke="#be123c" stroke-width="1.1899013136487278"/><path d="M967.0 124 L830 208 M835.816633008165 199.86570344477622 L830 208 L839.8876744216715 206.50537813109042" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><text x="170.5" y="152" font-size="22" fill="#be123c" text-anchor="middle" font-weight="650">0.693</text><rect x="747" y="214" width="165" height="45" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="830" y="244" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">10 · the</text><text x="22" y="230" font-size="24" fill="#586174" text-anchor="start" font-weight="500">Thicker arrow = more attention weight</text><text x="946" y="243" font-size="22" fill="#586174" text-anchor="start" font-weight="500">receiver</text></svg>

The sentence and receiver stay fixed. A different head gives fisherman a large weight.



<a id="s01-v-both"></a>
## Keep both readings

[Matching slide](../../part3.html?present#s01/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 505" role="img" aria-label="Two different reading patterns" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Two different reading patterns</title><text x="20" y="32" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 1: setting clues</text><rect x="20" y="69" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="69" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="69" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="69" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="69" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="69" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="69" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="69" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="69" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="69" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="100" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="101" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><path d="M54.5 124 L830 208 M820.4235972807309 210.8796720227094 L830 208 L821.2623057087912 203.1365959993665" fill="none" stroke="#be123c" stroke-width="0.6077888739354684"/><path d="M170.5 124 L830 208 M820.3711805086315 210.69922863104298 L830 208 L821.3552287580618 202.97327838700437" fill="none" stroke="#be123c" stroke-width="2.04441104465339"/><path d="M284.5 124 L830 208 M820.3040180798132 210.4470256646408 L830 208 L821.4893555773796 202.74938751080228" fill="none" stroke="#be123c" stroke-width="0.6300031000084825"/><path d="M380.5 124 L830 208 M820.2307837513852 210.1359807788928 L830 208 L821.6614626727436 202.48014536043274" fill="none" stroke="#be123c" stroke-width="0.6607565449036875"/><path d="M476.0 124 L830 208 M820.1391557312623 209.66245309939978 L830 208 L821.9373127814537 202.08450553073624" fill="none" stroke="#be123c" stroke-width="0.6179947723052064"/><path d="M565.0 124 L830 208 M820.0432490444437 208.92903735609943 L830 208 L822.396614199894 201.50473061569065" fill="none" stroke="#be123c" stroke-width="7.008797895659618"/><path d="M664.0 124 L830 208 M820.0234214675627 207.31598188173805 L830 208 L823.5399353635429 200.36668061111052" fill="none" stroke="#be123c" stroke-width="1.0286863304536027"/><path d="M753.0 124 L830 208 M820.9055505868265 203.84175639587716 L830 208 L826.6467774528851 198.5789651019901" fill="none" stroke="#be123c" stroke-width="0.5991148633565428"/><path d="M859.5 124 L830 208 M829.3777520839247 198.0193783995715 L830 208 L836.7261353980238 200.6000606348801" fill="none" stroke="#be123c" stroke-width="0.7033317113674585"/><path d="M967.0 124 L830 208 M835.816633008165 199.86570344477622 L830 208 L839.8876744216715 206.50537813109042" fill="none" stroke="#be123c" stroke-width="0.5991148633565428"/><text x="565.0" y="152" font-size="22" fill="#be123c" text-anchor="middle" font-weight="650">0.718</text><rect x="747" y="214" width="165" height="45" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="830" y="244" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">10 · the</text><text x="22" y="230" font-size="24" fill="#586174" text-anchor="start" font-weight="500">Thicker arrow = more attention weight</text><text x="946" y="243" font-size="22" fill="#586174" text-anchor="start" font-weight="500">receiver</text><text x="20" y="269" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 2: person clues</text><rect x="20" y="306" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="306" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="306" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="306" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="306" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="306" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="306" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="306" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="306" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="306" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="337" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="338" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><path d="M54.5 361 L830 445 M820.4235972807309 447.87967202270937 L830 445 L821.2623057087912 440.13659599936653" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M170.5 361 L830 445 M820.3711805086315 447.69922863104296 L830 445 L821.3552287580618 439.97327838700437" fill="none" stroke="#be123c" stroke-width="6.786645910206812"/><path d="M284.5 361 L830 445 M820.3040180798132 447.44702566464076 L830 445 L821.4893555773796 439.7493875108023" fill="none" stroke="#be123c" stroke-width="1.0938532271216301"/><path d="M380.5 361 L830 445 M820.2307837513852 447.13598077889276 L830 445 L821.6614626727436 439.48014536043274" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M476.0 361 L830 445 M820.1391557312623 446.6624530993998 L830 445 L821.9373127814537 439.08450553073624" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><path d="M565.0 361 L830 445 M820.0432490444437 445.9290373560994 L830 445 L822.396614199894 438.5047306156906" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><path d="M664.0 361 L830 445 M820.0234214675627 444.315981881738 L830 445 L823.5399353635429 437.36668061111055" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M753.0 361 L830 445 M820.9055505868265 440.84175639587716 L830 445 L826.6467774528851 435.5789651019901" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M859.5 361 L830 445 M829.3777520839247 435.0193783995715 L830 445 L836.7261353980238 437.6000606348801" fill="none" stroke="#be123c" stroke-width="1.1899013136487278"/><path d="M967.0 361 L830 445 M835.816633008165 436.8657034447762 L830 445 L839.8876744216715 443.5053781310904" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><text x="170.5" y="389" font-size="22" fill="#be123c" text-anchor="middle" font-weight="650">0.693</text><rect x="747" y="451" width="165" height="45" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="830" y="481" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">10 · the</text><text x="22" y="467" font-size="24" fill="#586174" text-anchor="start" font-weight="500">Thicker arrow = more attention weight</text><text x="946" y="480" font-size="22" fill="#586174" text-anchor="start" font-weight="500">receiver</text></svg>

Two heads can retain different source mixtures at the same token. They run in parallel, not one after the other.



<a id="s02-v-break"></a>
## How does each head choose what to read?

[Matching slide](../../part3.html?present#s02/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 325" role="img" aria-label="Queries, keys and values in each head" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Queries, keys and values in each head</title><text x="35" y="155" font-size="43" fill="#14171f" text-anchor="start" font-weight="650">Queries, keys and values in each head</text><text x="35" y="230" font-size="29" fill="#586174" text-anchor="start" font-weight="500">Each head learns its own W<tspan baseline-shift="sub" font-size="70%">Q</tspan>, W<tspan baseline-shift="sub" font-size="70%">K</tspan> and W<tspan baseline-shift="sub" font-size="70%">V</tspan>.</text></svg>



<p>Visual inspiration: <a href="https://www.3blue1brown.com/lessons/attention/">3Blue1Brown’s attention lesson</a> and <a href="https://jalammar.github.io/illustrated-transformer/">Jay Alammar’s Illustrated Transformer</a>. We keep our own river-bank example, numbers and Part II row-vector convention. A head-specific superscript labels a head; a subscript still labels a token.</p>

<a id="s02-v-recall"></a>
## Recall the Maya example: query, key and value

[Matching slide](../../part3.html?present#s02/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 435" role="img" aria-label="Recall Part II: the query asks, the key matches, the value supplies content" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Recall Part II: the query asks, the key matches, the value supplies content</title><text x="24" y="36" font-size="27" fill="#14171f" text-anchor="start" font-weight="500">Maya cycled home in the rain. Cold and tired, Maya reached</text><text x="24" y="76" font-size="27" fill="#14171f" text-anchor="start" font-weight="500">for a hooded red wool coat. She …</text><text x="24" y="148" font-size="27" fill="#8b2cde" text-anchor="start" font-weight="650">q: what is needed?</text><text x="400" y="148" font-size="26" fill="#8b2cde" text-anchor="start" font-weight="500">She × W<tspan baseline-shift="sub" font-size="70%">Q</tspan></text><path d="M637 139 L691 139 M681.7893900599712 142.8941834230865 L691 139 L681.7893900599712 135.1058165769135" fill="none" stroke="#8b2cde" stroke-width="2.5"/><text x="717" y="148" font-size="25" fill="#8b2cde" text-anchor="start" font-weight="500">Which earlier person?</text><text x="24" y="231" font-size="27" fill="#aa4e08" text-anchor="start" font-weight="650">k: what can match?</text><text x="400" y="231" font-size="26" fill="#aa4e08" text-anchor="start" font-weight="500">Maya × W<tspan baseline-shift="sub" font-size="70%">K</tspan></text><path d="M637 222 L691 222 M681.7893900599712 225.8941834230865 L691 222 L681.7893900599712 218.1058165769135" fill="none" stroke="#aa4e08" stroke-width="2.5"/><text x="717" y="231" font-size="25" fill="#aa4e08" text-anchor="start" font-weight="500">Person candidate</text><text x="24" y="314" font-size="27" fill="#0f766e" text-anchor="start" font-weight="650">v: what is sent?</text><text x="400" y="314" font-size="26" fill="#0f766e" text-anchor="start" font-weight="500">Maya × W<tspan baseline-shift="sub" font-size="70%">V</tspan></text><path d="M637 305 L691 305 M681.7893900599712 308.89418342308653 L691 305 L681.7893900599712 301.10581657691347" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="717" y="314" font-size="25" fill="#0f766e" text-anchor="start" font-weight="500">Cold, tired; cycled in rain</text><text x="580" y="399" font-size="25" fill="#586174" text-anchor="middle" font-weight="500">“Maya” and “She” here mean their current embedding rows.</text></svg>

In Part II’s illustrative later-layer example, the query asks for a person. Maya’s key can match that request. Her value supplies useful details. A head uses separate projections for these roles.

<p>This is the verbal example from <a href="../../attention.html#s11-frame-separate-maps">Part II</a>. It illustrates possible later-layer representations, not measured model outputs. Earlier layers may have gathered the preceding facts into the second Maya row. Every token has a query, key and value, even though this example follows only She’s query and Maya’s key/value. The actual vectors contain numbers, not written questions or records.</p>

<a id="s02-v-plan"></a>
## The two heads and the output projection

[Matching slide](../../part3.html?present#s02/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 455" role="img" aria-label="The same next-token path, with two parallel heads" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The same next-token path, with two parallel heads</title><rect x="23" y="136" width="177" height="69" rx="4" fill="#245edb08" stroke="#245edb" stroke-width="1.5"/><text x="111.5" y="164" font-size="25" fill="#245edb" text-anchor="middle" font-weight="600">Input E</text><text x="111.5" y="189" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 rows × 4</text><path d="M200 170 H238 V90 H285" fill="none" stroke="#245edb" stroke-width="2.5"/><rect x="285" y="55" width="235" height="69" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="83" font-size="25" fill="#8b2cde" text-anchor="middle" font-weight="600">Head 1</text><text x="402.5" y="108" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">Q, K, V → A → AV</text><path d="M520 90 L602 90 M592.7893900599712 93.8941834230865 L602 90 L592.7893900599712 86.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="604" y="55" width="182" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="695.0" y="83" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">H<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="695.0" y="108" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 messages × 2</text><path d="M786 90 H821 V149 H867" fill="none" stroke="#0f766e" stroke-width="2.5"/><path d="M200 170 H238 V264 H285" fill="none" stroke="#245edb" stroke-width="2.5"/><rect x="285" y="229" width="235" height="69" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="257" font-size="25" fill="#8b2cde" text-anchor="middle" font-weight="600">Head 2</text><text x="402.5" y="282" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">Q, K, V → A → AV</text><path d="M520 264 L602 264 M592.7893900599712 267.89418342308653 L602 264 L592.7893900599712 260.10581657691347" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="604" y="229" width="182" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="695.0" y="257" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">H<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="695.0" y="282" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 messages × 2</text><path d="M786 264 H821 V149 H867" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="867" y="115" width="265" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="999.5" y="143" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">Concatenate</text><text x="999.5" y="168" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">two [10×2] → [10×4]</text><path d="M1000 184 L1000 222 M996.1058165769135 212.78939005997114 L1000 222 L1003.8941834230865 212.78939005997114" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="867" y="226" width="265" height="69" rx="4" fill="#14773708" stroke="#147737" stroke-width="1.5"/><text x="999.5" y="254" font-size="25" fill="#147737" text-anchor="middle" font-weight="600">Project with W<tspan baseline-shift="sub" font-size="70%">O</tspan></text><text x="999.5" y="279" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">[10×4] × [4×4]</text><path d="M110 136 V40 H1148 V379 H1046" fill="none" stroke="#245edb" stroke-width="2" stroke-dasharray="7 6"/><path d="M1046 379 L1025 379 M1034.210609940029 375.10581657691347 L1025 379 L1034.210609940029 382.89418342308653" fill="none" stroke="#245edb" stroke-width="2.5"/><text x="980" y="25" font-size="20" fill="#245edb" text-anchor="middle" font-weight="500">keep original E</text><path d="M1000 295 L1000 353 M996.1058165769135 343.78939005997114 L1000 353 L1003.8941834230865 343.78939005997114" fill="none" stroke="#147737" stroke-width="2.5"/><text x="986" y="329" font-size="21" fill="#147737" text-anchor="end" font-weight="500">ΔE [10×4]</text><circle cx="1000" cy="379" r="24" fill="white" stroke="#147737" stroke-width="2"/><text x="1000" y="388" font-size="30" fill="#147737" text-anchor="middle" font-weight="500">+</text><path d="M973 379 L822 379 M831.2106099400288 375.10581657691347 L822 379 L831.2106099400288 382.89418342308653" fill="none" stroke="#147737" stroke-width="2.5"/><rect x="608" y="345" width="210" height="69" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="713.0" y="373" font-size="25" fill="#586174" text-anchor="middle" font-weight="600">E′ = E + ΔE</text><text x="713.0" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">same 10 × 4 shape</text><path d="M608 380 L520 380 M529.2106099400288 376.10581657691347 L520 380 L529.2106099400288 383.89418342308653" fill="none" stroke="#147737" stroke-width="2.5"/><rect x="285" y="345" width="235" height="69" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="402.5" y="373" font-size="25" fill="#586174" text-anchor="middle" font-weight="600">last row → MLP</text><text x="402.5" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">next-token prediction</text></svg>

Both heads read \(E\). Concatenation joins their message coordinates. \(W_O\) projects the joined messages into embedding space before we add the update to \(E\).

<p>H¹ and H² each have shape [10, 2]. Concatenation gives [10, 4]. Multiplying by W_O [4, 4] gives ΔE [10, 4]. Here the joined width already equals the embedding width. The projection still learns how to mix head outputs into embedding coordinates; it need not change the width. In general W_O has shape [n_heads × d_v, d_model]. It is not an inverse of the input projections.</p>

<a id="s02-v-roles"></a>
## The same roles in the river example

[Matching slide](../../part3.html?present#s02/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 442" role="img" aria-label="The familiar query, key and value roles, now repeated in two heads" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The familiar query, key and value roles, now repeated in two heads</title><text x="580" y="36" font-size="28" fill="#245edb" text-anchor="middle" font-weight="500">Receiver: the final “the” in the river-bank prefix</text><text x="24" y="109" font-size="23" fill="#586174" text-anchor="start" font-weight="600">Head</text><text x="230" y="109" font-size="23" fill="#586174" text-anchor="start" font-weight="600">Query asks about</text><text x="575" y="109" font-size="23" fill="#586174" text-anchor="start" font-weight="600">A useful source</text><text x="894" y="109" font-size="23" fill="#586174" text-anchor="start" font-weight="600">Value carries</text><text x="24" y="179" font-size="32" fill="#14171f" text-anchor="start" font-weight="500">1</text><text x="230" y="179" font-size="29" fill="#8b2cde" text-anchor="start" font-weight="500">the setting</text><text x="575" y="179" font-size="29" fill="#aa4e08" text-anchor="start" font-weight="500">river</text><text x="894" y="179" font-size="27" fill="#0f766e" text-anchor="start" font-weight="500">setting clues</text><text x="24" y="279" font-size="32" fill="#14171f" text-anchor="start" font-weight="500">2</text><text x="230" y="279" font-size="29" fill="#8b2cde" text-anchor="start" font-weight="500">the person</text><text x="575" y="279" font-size="29" fill="#aa4e08" text-anchor="start" font-weight="500">fisherman</text><text x="894" y="279" font-size="27" fill="#0f766e" text-anchor="start" font-weight="500">person clues</text><text x="580" y="365" font-size="28" fill="#14171f" text-anchor="middle" font-weight="500">Each head computes q, k and v for every token.</text><text x="580" y="406" font-size="25" fill="#586174" text-anchor="middle" font-weight="500">We follow one receiver and two useful sources.</text></svg>

Head 1 can ask about the setting; head 2 can ask about the person. These are interpretations of our chosen numbers. Training learns the projections without assigning these jobs.



<a id="s02-v-query"></a>
## Head 1: computing the setting query

[Matching slide](../../part3.html?present#s02/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 445" role="img" aria-label="Head 1: multiply the same four-coordinate embedding by its own query projection" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Head 1: multiply the same four-coordinate embedding by its own query projection</title><text x="24" y="43" font-size="27" fill="#14171f" text-anchor="start" font-weight="650">Receiver: final “the”</text><text x="620" y="43" font-size="31" fill="#8b2cde" text-anchor="start" font-weight="500">q₁₀<tspan baseline-shift="super" font-size="65%">(1)</tspan> = e₁₀ W<tspan baseline-shift="sub" font-size="70%">Q</tspan><tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="24" y="122" font-size="25" fill="#245edb" text-anchor="start" font-weight="500">e₁₀: four input coordinates</text><rect x="24" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="69.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.0</text><text x="69.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">water</text><rect x="115" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="160.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.0</text><text x="160.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">finance</text><rect x="206" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="251.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.0</text><text x="251.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">person</text><rect x="297" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="342.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">2.3</text><text x="342.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">glue</text><text x="419" y="202" font-size="37" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="678" y="92" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">Q</tspan><tspan baseline-shift="super" font-size="65%">(1)</tspan>  [4 × 2]</text><text x="626" y="131" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="500">water?</text><text x="730" y="131" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="500">finance?</text><rect x="574" y="149" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="174.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="678" y="149" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="174.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="574" y="188" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="213.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="678" y="188" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="213.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="574" y="227" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="252.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="678" y="227" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="252.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="574" y="266" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="291.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">1</text><rect x="678" y="266" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="291.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">1</text><path d="M803 216 L869 216 M859.7893900599712 219.8941834230865 L869 216 L859.7893900599712 212.1058165769135" fill="none" stroke="#8b2cde" stroke-width="2.5"/><text x="1000" y="132" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">q₁₀<tspan baseline-shift="super" font-size="65%">(1)</tspan>  [1 × 2]</text><rect x="890" y="167" width="110" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="945.0" y="199.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="1000" y="167" width="110" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="1055.0" y="199.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><text x="580" y="359" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">First query coordinate: 0×0 + 0×0 + 0×0 + 2.3×1 = 2.3</text><text x="580" y="403" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">Second query coordinate: 0×0 + 0×0 + 0×0 + 2.3×1 = 2.3</text></svg>

In this toy, the final “the” has only a nonzero glue coordinate. The two columns of \(W_Q\) turn that input into water and finance matching features.

<p>E contains position-aware embedding rows, not token IDs. The water, finance, person and glue axes are invented for teaching; glue is our toy feature for function words such as “the”. W_Q reads all four input coordinates. Each query coordinate is a dot product with one column of W_Q. These sparse matrices are hand-chosen; learned projections need not be sparse or interpretable. The head superscripts label heads, not powers. Query/key width is two per head here; Part II’s single-head toy used three.</p>

<a id="s02-v-query-person"></a>
## Head 2: computing the person query

[Matching slide](../../part3.html?present#s02/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 445" role="img" aria-label="Head 2: multiply the same four-coordinate embedding by its own query projection" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Head 2: multiply the same four-coordinate embedding by its own query projection</title><text x="24" y="43" font-size="27" fill="#14171f" text-anchor="start" font-weight="650">Receiver: final “the”</text><text x="620" y="43" font-size="31" fill="#8b2cde" text-anchor="start" font-weight="500">q₁₀<tspan baseline-shift="super" font-size="65%">(2)</tspan> = e₁₀ W<tspan baseline-shift="sub" font-size="70%">Q</tspan><tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="24" y="122" font-size="25" fill="#245edb" text-anchor="start" font-weight="500">e₁₀: four input coordinates</text><rect x="24" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="69.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.0</text><text x="69.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">water</text><rect x="115" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="160.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.0</text><text x="160.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">finance</text><rect x="206" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="251.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.0</text><text x="251.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">person</text><rect x="297" y="167" width="91" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="342.5" y="199.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">2.3</text><text x="342.5" y="155" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">glue</text><text x="419" y="202" font-size="37" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="678" y="92" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">Q</tspan><tspan baseline-shift="super" font-size="65%">(2)</tspan>  [4 × 2]</text><text x="626" y="131" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="500">person?</text><text x="730" y="131" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="500">glue?</text><rect x="574" y="149" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="174.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="678" y="149" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="174.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="574" y="188" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="213.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="678" y="188" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="213.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="574" y="227" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="252.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="678" y="227" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="252.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><rect x="574" y="266" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="626.0" y="291.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">1</text><rect x="678" y="266" width="104" height="39" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="730.0" y="291.35" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0</text><path d="M803 216 L869 216 M859.7893900599712 219.8941834230865 L869 216 L859.7893900599712 212.1058165769135" fill="none" stroke="#8b2cde" stroke-width="2.5"/><text x="1000" y="132" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">q₁₀<tspan baseline-shift="super" font-size="65%">(2)</tspan>  [1 × 2]</text><rect x="890" y="167" width="110" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="945.0" y="199.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="1000" y="167" width="110" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="1055.0" y="199.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0.0</text><text x="580" y="359" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">First query coordinate: 0×0 + 0×0 + 0×0 + 2.3×1 = 2.3</text><text x="580" y="403" font-size="27" fill="#8b2cde" text-anchor="middle" font-weight="500">Second query coordinate: 0×0 + 0×0 + 0×0 + 2.3×0 = 0.0</text></svg>

The input embedding stays [0, 0, 0, 2.3]. Head 2 uses its own \(W_Q\). Its query requests the person feature and gives zero weight to the glue matching feature.



<a id="s02-v-sources"></a>
## Source rows supply keys and values

[Matching slide](../../part3.html?present#s02/7/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 445" role="img" aria-label="Each source embedding supplies a key for matching and a value for the weighted message" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Each source embedding supplies a key for matching and a value for the weighted message</title><text x="24" y="35" font-size="26" fill="#245edb" text-anchor="start" font-weight="500">Source input eⱼ: [water, finance, person, glue]</text><text x="24" y="102" font-size="28" fill="#14171f" text-anchor="start" font-weight="650">Head 1: river</text><text x="24" y="146" font-size="27" fill="#245edb" text-anchor="start" font-weight="500">[3.1, −0.1, 0.0, 0.1]</text><text x="465" y="102" font-size="24" fill="#aa4e08" text-anchor="start" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">K</tspan><tspan baseline-shift="super" font-size="65%">(1)</tspan> selects water, finance</text><text x="465" y="146" font-size="24" fill="#0f766e" text-anchor="start" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">V</tspan><tspan baseline-shift="super" font-size="65%">(1)</tspan> selects water, finance</text><text x="884" y="102" font-size="25" fill="#aa4e08" text-anchor="start" font-weight="500">k = [3.1, −0.1]</text><text x="884" y="146" font-size="25" fill="#0f766e" text-anchor="start" font-weight="500">v = [3.1, −0.1]</text><text x="24" y="266" font-size="28" fill="#14171f" text-anchor="start" font-weight="650">Head 2: fisherman</text><text x="24" y="310" font-size="27" fill="#245edb" text-anchor="start" font-weight="500">[2.0, 0.1, 2.1, 0.0]</text><text x="465" y="266" font-size="24" fill="#aa4e08" text-anchor="start" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">K</tspan><tspan baseline-shift="super" font-size="65%">(2)</tspan> selects person, glue</text><text x="465" y="310" font-size="24" fill="#0f766e" text-anchor="start" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">V</tspan><tspan baseline-shift="super" font-size="65%">(2)</tspan> selects person, glue</text><text x="884" y="266" font-size="25" fill="#aa4e08" text-anchor="start" font-weight="500">k = [2.1, 0.0]</text><text x="884" y="310" font-size="25" fill="#0f766e" text-anchor="start" font-weight="500">v = [2.1, 0.0]</text><text x="580" y="409" font-size="27" fill="#14171f" text-anchor="middle" font-weight="500">kⱼ = eⱼ W<tspan baseline-shift="sub" font-size="70%">K</tspan> sets the match.  vⱼ = eⱼ W<tspan baseline-shift="sub" font-size="70%">V</tspan> supplies the message.</text></svg>

Each head has its own \(W_K\) and \(W_V\), both \(4\times2\) here. For easy arithmetic, they select the same coordinates. Keys determine matching; values carry the numbers we mix.

<p>Head 1 uses W_K = W_V = [[1,0],[0,1],[0,0],[0,0]], selecting water and finance. Head 2 uses W_K = W_V = [[0,0],[0,0],[1,0],[0,1]], selecting person and glue. Each matrix is [4,2] and acts on every source row, including sources not shown here. W_K and W_V are distinct parameters with equal numerical entries in this worksheet. They need not be equal in a trained model. In Part II’s Maya example they served different roles too.</p>

In [3]:
# Every projected row is computed from an input embedding.
case = worksheet['headsLesson']['cases']['river']
E = torch.tensor(case['E'])
for h, projection in enumerate(worksheet['headsLesson']['projections']):
    Q, K, V = [E @ torch.tensor(projection[kind], dtype=torch.float32)
               for kind in ['Q', 'K', 'V']]
    source = 5 if h == 0 else 1  # river or fisherman, zero-based index
    print(f'Head {h+1}: final query', Q[-1].tolist())
    print('Source:', case['tokens'][source],
          'key:', K[source].tolist(), 'value:', V[source].tolist())
    for kind, actual in [('Q', Q), ('K', K), ('V', V)]:
        torch.testing.assert_close(actual, torch.tensor(case['heads'][h][kind]))

Head 1: final query [2.299999952316284, 2.299999952316284]
Source: river key: [3.0999999046325684, -0.10000000149011612] value: [3.0999999046325684, -0.10000000149011612]
Head 2: final query [2.299999952316284, 0.0]
Source: fisherman key: [2.0999999046325684, 0.0] value: [2.0999999046325684, 0.0]


<a id="s02-v-match"></a>
## Each query meets keys from its own head

[Matching slide](../../part3.html?present#s02/8/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 416" role="img" aria-label="Compute a query-key score separately within each head" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Compute a query-key score separately within each head</title><text x="24" y="52" font-size="29" fill="#14171f" text-anchor="start" font-weight="650">Head 1</text><text x="225" y="52" font-size="22" fill="#8b2cde" text-anchor="start" font-weight="500">query</text><text x="455" y="52" font-size="22" fill="#aa4e08" text-anchor="start" font-weight="500">river key</text><rect x="224" y="69" width="86" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="267.0" y="101.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="310" y="69" width="86" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="353.0" y="101.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><text x="421" y="103" font-size="31" fill="#14171f" text-anchor="middle" font-weight="500">·</text><rect x="451" y="69" width="86" height="50" rx="4" fill="#aa4e080b" stroke="#aa4e08" stroke-width="1.5"/><text x="494.0" y="101.5" font-size="26" fill="#aa4e08" text-anchor="middle" font-weight="500">3.1</text><rect x="537" y="69" width="86" height="50" rx="4" fill="#aa4e080b" stroke="#aa4e08" stroke-width="1.5"/><text x="580.0" y="101.5" font-size="26" fill="#aa4e08" text-anchor="middle" font-weight="500">−0.1</text><text x="665" y="100" font-size="28" fill="#586174" text-anchor="start" font-weight="500">÷ √2</text><path d="M756 92 L817 92 M807.7893900599712 95.8941834230865 L817 92 L807.7893900599712 88.1058165769135" fill="none" stroke="#586174" stroke-width="2.5"/><text x="846" y="100" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">4.879</text><text x="224" y="163" font-size="24" fill="#586174" text-anchor="start" font-weight="500">(2.3 × 3.1 + 2.3 × −0.1) / √2</text><text x="24" y="239" font-size="29" fill="#14171f" text-anchor="start" font-weight="650">Head 2</text><text x="225" y="239" font-size="22" fill="#8b2cde" text-anchor="start" font-weight="500">query</text><text x="455" y="239" font-size="22" fill="#aa4e08" text-anchor="start" font-weight="500">fisherman key</text><rect x="224" y="256" width="86" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="267.0" y="288.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="310" y="256" width="86" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="353.0" y="288.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0.0</text><text x="421" y="290" font-size="31" fill="#14171f" text-anchor="middle" font-weight="500">·</text><rect x="451" y="256" width="86" height="50" rx="4" fill="#aa4e080b" stroke="#aa4e08" stroke-width="1.5"/><text x="494.0" y="288.5" font-size="26" fill="#aa4e08" text-anchor="middle" font-weight="500">2.1</text><rect x="537" y="256" width="86" height="50" rx="4" fill="#aa4e080b" stroke="#aa4e08" stroke-width="1.5"/><text x="580.0" y="288.5" font-size="26" fill="#aa4e08" text-anchor="middle" font-weight="500">0.0</text><text x="665" y="287" font-size="28" fill="#586174" text-anchor="start" font-weight="500">÷ √2</text><path d="M756 279 L817 279 M807.7893900599712 282.89418342308653 L817 279 L807.7893900599712 275.10581657691347" fill="none" stroke="#586174" stroke-width="2.5"/><text x="846" y="287" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">3.415</text><text x="224" y="350" font-size="24" fill="#586174" text-anchor="start" font-weight="500">(2.3 × 2.1 + 0.0 × 0.0) / √2</text></svg>

Head 1 exposes water and finance in its keys. Head 2 exposes person and glue. Score every source with the matching query.



<a id="s02-v-weights"></a>
## Each head gets its own weight row

[Matching slide](../../part3.html?present#s02/9/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 381" role="img" aria-label="Each head normalizes its own scores over the same source positions" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Each head normalizes its own scores over the same source positions</title><text x="15" y="100" font-size="25" fill="#14171f" text-anchor="start" font-weight="650">Head 1</text><text x="190.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">The</text><rect x="155" y="75" width="70" height="49" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="1.5"/><text x="190.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.006</text><text x="303.5" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><rect x="235" y="75" width="137" height="49" rx="4" fill="#be123c2d" stroke="#d9dfe9" stroke-width="1.5"/><text x="303.5" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.166</text><text x="417.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><rect x="382" y="75" width="70" height="49" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="1.5"/><text x="417.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.009</text><text x="512.5" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><rect x="462" y="75" width="101" height="49" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="1.5"/><text x="512.5" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.012</text><text x="608.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">the</text><rect x="573" y="75" width="70" height="49" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="1.5"/><text x="608.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.008</text><text x="697.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">river</text><rect x="653" y="75" width="88" height="49" rx="4" fill="#be123c88" stroke="#d9dfe9" stroke-width="1.5"/><text x="697.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.718</text><text x="793.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><rect x="751" y="75" width="84" height="49" rx="4" fill="#be123c1a" stroke="#d9dfe9" stroke-width="1.5"/><text x="793.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.053</text><text x="880.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">and</text><rect x="845" y="75" width="70" height="49" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="1.5"/><text x="880.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.005</text><text x="984.5" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><rect x="925" y="75" width="119" height="49" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="1.5"/><text x="984.5" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.017</text><text x="1089.0" y="55" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">the</text><rect x="1054" y="75" width="70" height="49" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="1.5"/><text x="1089.0" y="108" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.005</text><text x="590" y="166" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">row sum = 1</text><text x="15" y="265" font-size="25" fill="#14171f" text-anchor="start" font-weight="650">Head 2</text><text x="190.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">The</text><rect x="155" y="240" width="70" height="49" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="1.5"/><text x="190.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.027</text><text x="303.5" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><rect x="235" y="240" width="137" height="49" rx="4" fill="#be123c84" stroke="#d9dfe9" stroke-width="1.5"/><text x="303.5" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.693</text><text x="417.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><rect x="382" y="240" width="70" height="49" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="1.5"/><text x="417.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.060</text><text x="512.5" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><rect x="462" y="240" width="101" height="49" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="1.5"/><text x="512.5" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.027</text><text x="608.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">the</text><rect x="573" y="240" width="70" height="49" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="1.5"/><text x="608.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.023</text><text x="697.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">river</text><rect x="653" y="240" width="88" height="49" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="1.5"/><text x="697.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.023</text><text x="793.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><rect x="751" y="240" width="84" height="49" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="1.5"/><text x="793.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.027</text><text x="880.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">and</text><rect x="845" y="240" width="70" height="49" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="1.5"/><text x="880.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.027</text><text x="984.5" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><rect x="925" y="240" width="119" height="49" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="1.5"/><text x="984.5" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.071</text><text x="1089.0" y="220" font-size="19" fill="#14171f" text-anchor="middle" font-weight="500">the</text><rect x="1054" y="240" width="70" height="49" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="1.5"/><text x="1089.0" y="273" font-size="23" fill="#be123c" text-anchor="middle" font-weight="500">0.023</text><text x="590" y="331" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">row sum = 1</text></svg>

Softmax runs across source positions, separately for each head. As before, \(\alpha_{ij}\) is one weight and \(A\) is the whole weight matrix.



<a id="s02-v-message"></a>
## Values turn those weights into messages

[Matching slide](../../part3.html?present#s02/10/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 393" role="img" aria-label="Each head weights its own value rows to obtain its message" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Each head weights its own value rows to obtain its message</title><text x="24" y="38" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 1 · river contributes</text><text x="43" y="104" font-size="29" fill="#be123c" text-anchor="start" font-weight="500">0.718</text><text x="150" y="104" font-size="30" fill="#14171f" text-anchor="start" font-weight="500">×</text><rect x="186" y="69" width="97" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="234.5" y="101.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">3.1</text><rect x="283" y="69" width="97" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="331.5" y="101.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">−0.1</text><text x="420" y="105" font-size="30" fill="#14171f" text-anchor="start" font-weight="500">+</text><text x="470" y="91" font-size="23" fill="#586174" text-anchor="start" font-weight="500">other weighted</text><text x="470" y="121" font-size="23" fill="#586174" text-anchor="start" font-weight="500">value rows</text><path d="M679 97 L753 97 M743.7893900599712 100.8941834230865 L753 97 L743.7893900599712 93.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="788" y="69" width="147" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="861.5" y="101.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">2.617</text><rect x="935" y="69" width="147" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="1008.5" y="101.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">−0.018</text><text x="938" y="38" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">m₁₀<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="234" y="147" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">water</text><text x="861" y="147" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">water</text><text x="331" y="147" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">finance</text><text x="1008" y="147" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">finance</text><text x="24" y="233" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 2 · fisherman contributes</text><text x="43" y="299" font-size="29" fill="#be123c" text-anchor="start" font-weight="500">0.693</text><text x="150" y="299" font-size="30" fill="#14171f" text-anchor="start" font-weight="500">×</text><rect x="186" y="264" width="97" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="234.5" y="296.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">2.1</text><rect x="283" y="264" width="97" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="331.5" y="296.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">0.0</text><text x="420" y="300" font-size="30" fill="#14171f" text-anchor="start" font-weight="500">+</text><text x="470" y="286" font-size="23" fill="#586174" text-anchor="start" font-weight="500">other weighted</text><text x="470" y="316" font-size="23" fill="#586174" text-anchor="start" font-weight="500">value rows</text><path d="M679 292 L753 292 M743.7893900599712 295.89418342308653 L753 292 L743.7893900599712 288.10581657691347" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="788" y="264" width="147" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="861.5" y="296.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">1.552</text><rect x="935" y="264" width="147" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="1008.5" y="296.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">0.538</text><text x="938" y="233" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">m₁₀<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="234" y="342" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">person</text><text x="861" y="342" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">person</text><text x="331" y="342" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">glue</text><text x="1008" y="342" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">glue</text></svg>

Keys set the weights; values carry the content. Here we chose equal key/value numbers, but their learned projections are separate.

<p>In this particular worksheet W_K and W_V have equal numerical entries within each head, so the displayed key and value numbers coincide. They remain separate parameters with different roles: keys set scores; values are mixed into messages. All ten source contributions, including those not expanded on the slide, enter each result.</p>

<a id="s02-v-wide"></a>
## What if we made one head wider?

[Matching slide](../../part3.html?present#s02/11/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 420" role="img" aria-label="A four-coordinate dot product adds both matching contributions before one softmax" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>A four-coordinate dot product adds both matching contributions before one softmax</title><text x="580" y="36" font-size="27" fill="#14171f" text-anchor="middle" font-weight="500">Join the same query and key coordinates into one wider head.</text><text x="70" y="92" font-size="28" fill="#8b2cde" text-anchor="start" font-weight="500">q₁₀ =</text><rect x="228" y="57" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="300.5" y="89.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="373" y="57" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="445.5" y="89.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="518" y="57" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="590.5" y="89.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="663" y="57" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="735.5" y="89.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0.0</text><path d="M518 53 V113" fill="none" stroke="#14171f" stroke-width="2"/><text x="374" y="144" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">setting coordinates</text><text x="663" y="144" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">person coordinates</text><text x="25" y="214" font-size="28" fill="#aa4e08" text-anchor="start" font-weight="650">river</text><text x="264" y="214" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">6.90</text><text x="403" y="214" font-size="30" fill="#14171f" text-anchor="start" font-weight="500">+</text><text x="464" y="214" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">0.00</text><text x="619" y="214" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">= 6.90</text><path d="M782 204 L904 204 M894.7893900599712 207.8941834230865 L904 204 L894.7893900599712 200.1058165769135" fill="none" stroke="#be123c" stroke-width="2.5"/><text x="842" y="185" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">÷ √4</text><text x="945" y="214" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">3.45</text><text x="25" y="319" font-size="28" fill="#aa4e08" text-anchor="start" font-weight="650">fisherman</text><text x="264" y="319" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">4.83</text><text x="403" y="319" font-size="30" fill="#14171f" text-anchor="start" font-weight="500">+</text><text x="464" y="319" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">4.83</text><text x="619" y="319" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">= 9.66</text><path d="M782 309 L904 309 M894.7893900599712 312.89418342308653 L904 309 L894.7893900599712 305.10581657691347" fill="none" stroke="#be123c" stroke-width="2.5"/><text x="842" y="290" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">÷ √4</text><text x="945" y="319" font-size="30" fill="#be123c" text-anchor="start" font-weight="500">4.83</text><text x="325" y="385" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">setting dot</text><text x="515" y="385" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">person dot</text><text x="990" y="385" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">one score</text></svg>

A wider query/key can compare more features. Their products still add into one score per source, followed by one softmax.

<p>For this controlled comparison, concatenate the two existing Q, K and V projections into width-four projections. A single wide head scales its dot products by √4. The two narrower heads each scale by √2 and normalize separately. The example changes no projection entries and does not compare separately trained models.</p>

<a id="s02-v-separate"></a>
## Two heads keep separate source preferences

[Matching slide](../../part3.html?present#s02/12/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 423" role="img" aria-label="The same projection coordinates yield one attention row or two separately normalized rows" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The same projection coordinates yield one attention row or two separately normalized rows</title><text x="22" y="48" font-size="23" fill="#586174" text-anchor="start" font-weight="600">Receiver 10 reads</text><text x="350" y="48" font-size="23" fill="#586174" text-anchor="start" font-weight="600">river</text><text x="570" y="48" font-size="23" fill="#586174" text-anchor="start" font-weight="600">fisherman</text><text x="790" y="48" font-size="23" fill="#586174" text-anchor="start" font-weight="600">others</text><text x="1010" y="48" font-size="23" fill="#586174" text-anchor="start" font-weight="600">value width</text><text x="22" y="127" font-size="27" fill="#14171f" text-anchor="start" font-weight="500">One wide head</text><rect x="346" y="91" width="158" height="56" rx="4" fill="#be123c2b" stroke="#be123c" stroke-width="1.5"/><text x="425" y="127" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.178</text><rect x="566" y="91" width="158" height="56" rx="4" fill="#be123c80" stroke="#be123c" stroke-width="1.5"/><text x="645" y="127" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.708</text><rect x="786" y="91" width="158" height="56" rx="4" fill="#be123c21" stroke="#be123c" stroke-width="1.5"/><text x="865" y="127" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.113</text><text x="1060" y="127" font-size="29" fill="#0f766e" text-anchor="middle" font-weight="500">4</text><text x="22" y="221" font-size="27" fill="#14171f" text-anchor="start" font-weight="500">Head 1: setting</text><rect x="346" y="185" width="158" height="56" rx="4" fill="#be123c81" stroke="#be123c" stroke-width="1.5"/><text x="425" y="221" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.718</text><rect x="566" y="185" width="158" height="56" rx="4" fill="#be123c29" stroke="#be123c" stroke-width="1.5"/><text x="645" y="221" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.166</text><rect x="786" y="185" width="158" height="56" rx="4" fill="#be123c21" stroke="#be123c" stroke-width="1.5"/><text x="865" y="221" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.116</text><text x="1060" y="221" font-size="29" fill="#0f766e" text-anchor="middle" font-weight="500">2</text><text x="22" y="315" font-size="27" fill="#14171f" text-anchor="start" font-weight="500">Head 2: person</text><rect x="346" y="279" width="158" height="56" rx="4" fill="#be123c12" stroke="#be123c" stroke-width="1.5"/><text x="425" y="315" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.023</text><rect x="566" y="279" width="158" height="56" rx="4" fill="#be123c7d" stroke="#be123c" stroke-width="1.5"/><text x="645" y="315" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.693</text><rect x="786" y="279" width="158" height="56" rx="4" fill="#be123c3c" stroke="#be123c" stroke-width="1.5"/><text x="865" y="315" font-size="27" fill="#be123c" text-anchor="middle" font-weight="500">0.284</text><text x="1060" y="315" font-size="29" fill="#0f766e" text-anchor="middle" font-weight="500">2</text><path d="M22 166 H1125" fill="none" stroke="#d9dfe9" stroke-width="2"/><text x="580" y="390" font-size="27" fill="#14171f" text-anchor="middle" font-weight="500">One shared mixture of four coordinates, or two separate mixtures of two.</text></svg>

A wider value vector carries more features with one shared weight row. Two heads can favour river for setting features and fisherman for person features.

<p>Every displayed attention row sums to one over all ten sources; “others” combines the remaining eight. One wide head uses its single row for all four value coordinates. The two heads use different rows for their two-coordinate values, preserving both mixtures before W_O combines them. This is a useful architectural choice, not proof that one wide head cannot learn useful relationships or that more heads always improve accuracy. See <a href="https://arxiv.org/abs/1706.03762">Attention Is All You Need, §3.2.2</a>.</p>

In [4]:
# Same projected coordinates, one wide softmax or two narrow softmaxes.
case = worksheet['headsLesson']['cases']['river']
Q_wide, K_wide, V_wide = [
    torch.cat([torch.tensor(head[kind]) for head in case['heads']], dim=-1)
    for kind in ['Q', 'K', 'V']
]
scores_wide = Q_wide[-1] @ K_wide.T / math.sqrt(4)
weights_wide = scores_wide.softmax(-1)  # all ten sources allowed at row 10
message_wide = weights_wide @ V_wide
torch.testing.assert_close(weights_wide, torch.tensor(case['wide']['A'][-1]))
torch.testing.assert_close(message_wide, torch.tensor(case['wide']['messages'][-1]))
for label, row in [('one wide head', weights_wide)] + [
    (f'head {h+1}', torch.tensor(head['A'][-1]))
    for h, head in enumerate(case['heads'])
]:
    print(label, 'river:', round(row[5].item(), 3),
          'fisherman:', round(row[1].item(), 3))

one wide head river: 0.178 fisherman: 0.708
head 1 river: 0.718 fisherman: 0.166
head 2 river: 0.023 fisherman: 0.693


<a id="s03-v-break"></a>
## What reaches the receiving token?

[Matching slide](../../part3.html?present#s03/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 325" role="img" aria-label="Combining the head messages" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Combining the head messages</title><text x="35" y="155" font-size="43" fill="#14171f" text-anchor="start" font-weight="650">Combining the head messages</text><text x="35" y="230" font-size="29" fill="#586174" text-anchor="start" font-weight="500">Concatenate, project into embedding space, then add the update.</text></svg>





<a id="s03-v-join"></a>
## Put the messages side by side

[Matching slide](../../part3.html?present#s03/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 375" role="img" aria-label="Concatenate two messages; do not average them" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Concatenate two messages; do not average them</title><text x="130" y="59" font-size="27" fill="#0f766e" text-anchor="start" font-weight="500">m₁₀<tspan baseline-shift="super" font-size="65%">(1)</tspan> · setting message</text><text x="718" y="59" font-size="27" fill="#0f766e" text-anchor="start" font-weight="500">m₁₀<tspan baseline-shift="super" font-size="65%">(2)</tspan> · person message</text><rect x="130" y="87" width="135" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="197.5" y="119.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">2.617</text><rect x="265" y="87" width="135" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="332.5" y="119.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">−0.018</text><rect x="718" y="87" width="135" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="785.5" y="119.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">1.552</text><rect x="853" y="87" width="135" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="920.5" y="119.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">0.538</text><path d="M265 144 L440 223 M430.00288977941307 222.7596102384942 L440 223 L433.2073912948401 215.66103093216864" fill="none" stroke="#0f766e" stroke-width="2.5"/><path d="M853 144 L728 223 M733.7055306541047 214.7873926213978 L728 223 L737.8664454123074 221.37111850462978" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="290" y="232" width="145" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="362.5" y="264.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">2.617</text><rect x="435" y="232" width="145" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="507.5" y="264.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">−0.018</text><rect x="580" y="232" width="145" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="652.5" y="264.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">1.552</text><rect x="725" y="232" width="145" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="797.5" y="264.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">0.538</text><path d="M580 227 V290" fill="none" stroke="#14171f" stroke-width="3"/><text x="580" y="333" font-size="29" fill="#14171f" text-anchor="middle" font-weight="500">2 coordinates + 2 coordinates → 4 coordinates</text></svg>

Concatenation keeps the two messages separate. It does not add them coordinate by coordinate.



<a id="s03-v-output"></a>
## Map the messages back to embedding space

[Matching slide](../../part3.html?present#s03/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 416" role="img" aria-label="Map the joined message back to the four input coordinates" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Map the joined message back to the four input coordinates</title><text x="98" y="64" font-size="25" fill="#0f766e" text-anchor="start" font-weight="500">joined message</text><rect x="20" y="99" width="112" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="76.0" y="131.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">2.617</text><rect x="132" y="99" width="112" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="188.0" y="131.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">−0.018</text><rect x="244" y="99" width="112" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="300.0" y="131.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">1.552</text><rect x="356" y="99" width="112" height="50" rx="4" fill="#0f766e0b" stroke="#0f766e" stroke-width="1.5"/><text x="412.0" y="131.5" font-size="26" fill="#0f766e" text-anchor="middle" font-weight="500">0.538</text><text x="491" y="136" font-size="36" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="730" y="29" font-size="27" fill="#147737" text-anchor="middle" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">O</tspan> · 4 × 4</text><rect x="566" y="50" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="604" y="81" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">1</text><rect x="641" y="50" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="679" y="81" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="716" y="50" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="754" y="81" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="791" y="50" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="829" y="81" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="566" y="97" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="604" y="128" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="641" y="97" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="679" y="128" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">1</text><rect x="716" y="97" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="754" y="128" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="791" y="97" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="829" y="128" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="566" y="144" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="604" y="175" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0.25</text><rect x="641" y="144" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="679" y="175" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="716" y="144" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="754" y="175" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">1</text><rect x="791" y="144" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="829" y="175" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="566" y="191" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="604" y="222" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="641" y="191" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="679" y="222" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="716" y="191" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="754" y="222" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">0</text><rect x="791" y="191" width="75" height="47" rx="4" fill="#14773709" stroke="#147737" stroke-width="1.5"/><text x="829" y="222" font-size="24" fill="#147737" text-anchor="middle" font-weight="500">1</text><path d="M560 144 H872" fill="none" stroke="#14171f" stroke-width="2.5"/><text x="914" y="103" font-size="21" fill="#586174" text-anchor="start" font-weight="500">head 1 rows</text><text x="914" y="199" font-size="21" fill="#586174" text-anchor="start" font-weight="500">head 2 rows</text><text x="126" y="312" font-size="32" fill="#147737" text-anchor="start" font-weight="500">Δe₁₀ =</text><rect x="300" y="277" width="156" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="378.0" y="309.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">3.005</text><rect x="456" y="277" width="156" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="534.0" y="309.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">−0.018</text><rect x="612" y="277" width="156" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="690.0" y="309.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">1.552</text><rect x="768" y="277" width="156" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="846.0" y="309.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">0.538</text><text x="580" y="377" font-size="26" fill="#14171f" text-anchor="middle" font-weight="500">First coordinate: 2.617 + 0.25 × 1.552 = 3.005</text></svg>

\(W_O\) learns how the head messages contribute to the embedding update. Four joined coordinates become four update coordinates.

<p>Equivalently, split W_O into a two-row block per head. Then Δe = m¹ W_O¹ + m² W_O²: each head contributes a four-coordinate update. Concatenation followed by one matrix multiplication computes exactly that sum. We retain the standard concat notation from the original Transformer paper.</p>

<a id="s03-v-residual"></a>
## Add context to the original embedding

[Matching slide](../../part3.html?present#s03/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 377" role="img" aria-label="Keep the original embedding row and add the context update" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Keep the original embedding row and add the context update</title><text x="23" y="89" font-size="30" fill="#245edb" text-anchor="start" font-weight="500">original e₁₀</text><rect x="360" y="54" width="181" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="450.5" y="86.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.000</text><rect x="541" y="54" width="181" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="631.5" y="86.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.000</text><rect x="722" y="54" width="181" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="812.5" y="86.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">0.000</text><rect x="903" y="54" width="181" height="50" rx="4" fill="#245edb0b" stroke="#245edb" stroke-width="1.5"/><text x="993.5" y="86.5" font-size="26" fill="#245edb" text-anchor="middle" font-weight="500">2.300</text><text x="450" y="37" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">water</text><text x="631" y="37" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">finance</text><text x="812" y="37" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">person</text><text x="993" y="37" font-size="23" fill="#586174" text-anchor="middle" font-weight="500">glue</text><text x="23" y="205" font-size="30" fill="#147737" text-anchor="start" font-weight="500">+ update Δe₁₀</text><rect x="360" y="170" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="450.5" y="202.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">3.005</text><rect x="541" y="170" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="631.5" y="202.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">−0.018</text><rect x="722" y="170" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="812.5" y="202.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">1.552</text><rect x="903" y="170" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="993.5" y="202.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">0.538</text><text x="23" y="321" font-size="30" fill="#147737" text-anchor="start" font-weight="500">= updated e′₁₀</text><rect x="360" y="286" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="450.5" y="318.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">3.005</text><rect x="541" y="286" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="631.5" y="318.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">−0.018</text><rect x="722" y="286" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="812.5" y="318.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">1.552</text><rect x="903" y="286" width="181" height="50" rx="4" fill="#1477370b" stroke="#147737" stroke-width="1.5"/><text x="993.5" y="318.5" font-size="26" fill="#147737" text-anchor="middle" font-weight="500">2.838</text></svg>

Exactly as in Part II: \(e_i\) is the input embedding row, \(\Delta e_i\) is the context update, and \(e_i^{\prime}=e_i+\Delta e_i\).



<a id="s03-v-map"></a>
## Return to the next-token prediction

[Matching slide](../../part3.html?present#s03/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 455" role="img" aria-label="The same next-token path, with two parallel heads" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The same next-token path, with two parallel heads</title><rect x="23" y="136" width="177" height="69" rx="4" fill="#245edb08" stroke="#245edb" stroke-width="1.5"/><text x="111.5" y="164" font-size="25" fill="#245edb" text-anchor="middle" font-weight="600">Input E</text><text x="111.5" y="189" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 rows × 4</text><path d="M200 170 H238 V90 H285" fill="none" stroke="#245edb" stroke-width="2.5"/><rect x="285" y="55" width="235" height="69" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="83" font-size="25" fill="#8b2cde" text-anchor="middle" font-weight="600">Head 1</text><text x="402.5" y="108" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">Q, K, V → A → AV</text><path d="M520 90 L602 90 M592.7893900599712 93.8941834230865 L602 90 L592.7893900599712 86.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="604" y="55" width="182" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="695.0" y="83" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">H<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="695.0" y="108" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 messages × 2</text><path d="M786 90 H821 V149 H867" fill="none" stroke="#0f766e" stroke-width="2.5"/><path d="M200 170 H238 V264 H285" fill="none" stroke="#245edb" stroke-width="2.5"/><rect x="285" y="229" width="235" height="69" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="257" font-size="25" fill="#8b2cde" text-anchor="middle" font-weight="600">Head 2</text><text x="402.5" y="282" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">Q, K, V → A → AV</text><path d="M520 264 L602 264 M592.7893900599712 267.89418342308653 L602 264 L592.7893900599712 260.10581657691347" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="604" y="229" width="182" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="695.0" y="257" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">H<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="695.0" y="282" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 messages × 2</text><path d="M786 264 H821 V149 H867" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="867" y="115" width="265" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="999.5" y="143" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">Concatenate</text><text x="999.5" y="168" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">two [10×2] → [10×4]</text><path d="M1000 184 L1000 222 M996.1058165769135 212.78939005997114 L1000 222 L1003.8941834230865 212.78939005997114" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="867" y="226" width="265" height="69" rx="4" fill="#14773708" stroke="#147737" stroke-width="1.5"/><text x="999.5" y="254" font-size="25" fill="#147737" text-anchor="middle" font-weight="600">Project with W<tspan baseline-shift="sub" font-size="70%">O</tspan></text><text x="999.5" y="279" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">[10×4] × [4×4]</text><path d="M110 136 V40 H1148 V379 H1046" fill="none" stroke="#245edb" stroke-width="2" stroke-dasharray="7 6"/><path d="M1046 379 L1025 379 M1034.210609940029 375.10581657691347 L1025 379 L1034.210609940029 382.89418342308653" fill="none" stroke="#245edb" stroke-width="2.5"/><text x="980" y="25" font-size="20" fill="#245edb" text-anchor="middle" font-weight="500">keep original E</text><path d="M1000 295 L1000 353 M996.1058165769135 343.78939005997114 L1000 353 L1003.8941834230865 343.78939005997114" fill="none" stroke="#147737" stroke-width="2.5"/><text x="986" y="329" font-size="21" fill="#147737" text-anchor="end" font-weight="500">ΔE [10×4]</text><circle cx="1000" cy="379" r="24" fill="white" stroke="#147737" stroke-width="2"/><text x="1000" y="388" font-size="30" fill="#147737" text-anchor="middle" font-weight="500">+</text><path d="M973 379 L822 379 M831.2106099400288 375.10581657691347 L822 379 L831.2106099400288 382.89418342308653" fill="none" stroke="#147737" stroke-width="2.5"/><rect x="608" y="345" width="210" height="69" rx="4" fill="#14773708" stroke="#147737" stroke-width="1.5"/><text x="713.0" y="373" font-size="25" fill="#147737" text-anchor="middle" font-weight="600">E′ = E + ΔE</text><text x="713.0" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">same 10 × 4 shape</text><path d="M608 380 L520 380 M529.2106099400288 376.10581657691347 L520 380 L529.2106099400288 383.89418342308653" fill="none" stroke="#147737" stroke-width="2.5"/><rect x="285" y="345" width="235" height="69" rx="4" fill="#245edb08" stroke="#245edb" stroke-width="1.5"/><text x="402.5" y="373" font-size="25" fill="#245edb" text-anchor="middle" font-weight="600">last row → MLP</text><text x="402.5" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">next-token prediction</text></svg>

The last updated row still feeds the prediction MLP. More heads change how it reads context—not what the target means.



<a id="s03-v-explore"></a>
## Try the other bank

[Matching slide](../../part3.html?present#s03/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 505" role="img" aria-label="Two different reading patterns" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Two different reading patterns</title><text x="20" y="32" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 1: setting clues</text><rect x="20" y="69" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="69" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="69" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="69" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="69" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="69" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="69" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="69" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="69" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="100" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="69" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="100" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="57" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="101" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><path d="M54.5 124 L830 208 M820.4235972807309 210.8796720227094 L830 208 L821.2623057087912 203.1365959993665" fill="none" stroke="#be123c" stroke-width="0.6077888739354684"/><path d="M170.5 124 L830 208 M820.3711805086315 210.69922863104298 L830 208 L821.3552287580618 202.97327838700437" fill="none" stroke="#be123c" stroke-width="2.04441104465339"/><path d="M284.5 124 L830 208 M820.3040180798132 210.4470256646408 L830 208 L821.4893555773796 202.74938751080228" fill="none" stroke="#be123c" stroke-width="0.6300031000084825"/><path d="M380.5 124 L830 208 M820.2307837513852 210.1359807788928 L830 208 L821.6614626727436 202.48014536043274" fill="none" stroke="#be123c" stroke-width="0.6607565449036875"/><path d="M476.0 124 L830 208 M820.1391557312623 209.66245309939978 L830 208 L821.9373127814537 202.08450553073624" fill="none" stroke="#be123c" stroke-width="0.6179947723052064"/><path d="M565.0 124 L830 208 M820.0432490444437 208.92903735609943 L830 208 L822.396614199894 201.50473061569065" fill="none" stroke="#be123c" stroke-width="7.008797895659618"/><path d="M664.0 124 L830 208 M820.0234214675627 207.31598188173805 L830 208 L823.5399353635429 200.36668061111052" fill="none" stroke="#be123c" stroke-width="1.0286863304536027"/><path d="M753.0 124 L830 208 M820.9055505868265 203.84175639587716 L830 208 L826.6467774528851 198.5789651019901" fill="none" stroke="#be123c" stroke-width="0.5991148633565428"/><path d="M859.5 124 L830 208 M829.3777520839247 198.0193783995715 L830 208 L836.7261353980238 200.6000606348801" fill="none" stroke="#be123c" stroke-width="0.7033317113674585"/><path d="M967.0 124 L830 208 M835.816633008165 199.86570344477622 L830 208 L839.8876744216715 206.50537813109042" fill="none" stroke="#be123c" stroke-width="0.5991148633565428"/><text x="565.0" y="152" font-size="22" fill="#be123c" text-anchor="middle" font-weight="650">0.718</text><rect x="747" y="214" width="165" height="45" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="830" y="244" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">10 · the</text><text x="22" y="230" font-size="24" fill="#586174" text-anchor="start" font-weight="500">Thicker arrow = more attention weight</text><text x="946" y="243" font-size="22" fill="#586174" text-anchor="start" font-weight="500">receiver</text><text x="20" y="269" font-size="26" fill="#14171f" text-anchor="start" font-weight="650">Head 2: person clues</text><rect x="20" y="306" width="69" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="54.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">The</text><text x="54.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">1</text><rect x="102" y="306" width="137" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="170.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">fisherman</text><text x="170.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">2</text><rect x="252" y="306" width="65" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="284.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">sat</text><text x="284.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">3</text><rect x="330" y="306" width="101" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="380.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">beside</text><text x="380.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">4</text><rect x="444" y="306" width="64" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="476.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">the</text><text x="476.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">5</text><rect x="521" y="306" width="88" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="565.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">river</text><text x="565.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">6</text><rect x="622" y="306" width="84" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="664.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">bank</text><text x="664.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">7</text><rect x="719" y="306" width="68" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="753.0" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">and</text><text x="753.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">8</text><rect x="800" y="306" width="119" height="48" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="859.5" y="337" font-size="23" fill="#14171f" text-anchor="middle" font-weight="500">watched</text><text x="859.5" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">9</text><rect x="932" y="306" width="70" height="48" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="967.0" y="337" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="650">the</text><text x="967.0" y="294" font-size="18" fill="#586174" text-anchor="middle" font-weight="500">10</text><text x="1120" y="338" font-size="25" fill="#147737" text-anchor="middle" font-weight="500">___</text><path d="M54.5 361 L830 445 M820.4235972807309 447.87967202270937 L830 445 L821.2623057087912 440.13659599936653" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M170.5 361 L830 445 M820.3711805086315 447.69922863104296 L830 445 L821.3552287580618 439.97327838700437" fill="none" stroke="#be123c" stroke-width="6.786645910206812"/><path d="M284.5 361 L830 445 M820.3040180798132 447.44702566464076 L830 445 L821.4893555773796 439.7493875108023" fill="none" stroke="#be123c" stroke-width="1.0938532271216301"/><path d="M380.5 361 L830 445 M820.2307837513852 447.13598077889276 L830 445 L821.6614626727436 439.48014536043274" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M476.0 361 L830 445 M820.1391557312623 446.6624530993998 L830 445 L821.9373127814537 439.08450553073624" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><path d="M565.0 361 L830 445 M820.0432490444437 445.9290373560994 L830 445 L822.396614199894 438.5047306156906" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><path d="M664.0 361 L830 445 M820.0234214675627 444.315981881738 L830 445 L823.5399353635429 437.36668061111055" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M753.0 361 L830 445 M820.9055505868265 440.84175639587716 L830 445 L826.6467774528851 435.5789651019901" fill="none" stroke="#be123c" stroke-width="0.7911710925013192"/><path d="M859.5 361 L830 445 M829.3777520839247 435.0193783995715 L830 445 L836.7261353980238 437.6000606348801" fill="none" stroke="#be123c" stroke-width="1.1899013136487278"/><path d="M967.0 361 L830 445 M835.816633008165 436.8657034447762 L830 445 L839.8876744216715 443.5053781310904" fill="none" stroke="#be123c" stroke-width="0.7549717263391844"/><text x="170.5" y="389" font-size="22" fill="#be123c" text-anchor="middle" font-weight="650">0.693</text><rect x="747" y="451" width="165" height="45" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="830" y="481" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">10 · the</text><text x="22" y="467" font-size="24" fill="#586174" text-anchor="start" font-weight="500">Thicker arrow = more attention weight</text><text x="946" y="480" font-size="22" fill="#586174" text-anchor="start" font-weight="500">receiver</text></svg>

Switch contexts in the matching slide to see both reading patterns change. The executable lab below computes the river and cheque examples from the same parameters.

<p>Change to the cheque sentence. These controls recalculate Q, K, V, both attention rows, the messages and the vocabulary prediction from the hand-chosen parameters. This is a worked arithmetic explorer, not a trained language model.</p>

<a id="s04-v-break"></a>
## Can every token do this at once?

[Matching slide](../../part3.html?present#s04/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 325" role="img" aria-label="The same calculation for every token" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The same calculation for every token</title><text x="35" y="155" font-size="43" fill="#14171f" text-anchor="start" font-weight="650">The same calculation for every token</text><text x="35" y="230" font-size="29" fill="#586174" text-anchor="start" font-weight="500">Trace the receiver row through the full matrices.</text></svg>





<a id="s04-v-project"></a>
## Project every row with the same head matrix

[Matching slide](../../part3.html?present#s04/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 420" role="img" aria-label="Project ten four-coordinate input rows into ten two-coordinate rows" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Project ten four-coordinate input rows into ten two-coordinate rows</title><text x="219.0" y="51" font-size="28" fill="#245edb" text-anchor="middle" font-weight="650">E</text><text x="219.0" y="79" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 4</text><rect x="135" y="93" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="93" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="93" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="93" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="117" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="117" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="117" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="117" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="141" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="141" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="141" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="141" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="165" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="165" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="165" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="165" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="189" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="189" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="189" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="189" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="213" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="213" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="213" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="213" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="237" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="237" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="237" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="237" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="261" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="261" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="261" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="261" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="285" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="285" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="285" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="285" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="309" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="177" y="309" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="219" y="309" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="261" y="309" width="42" height="24" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="135" y="309" width="168" height="24" rx="4" fill="none" stroke="#245edb" stroke-width="2.5"/><text x="365" y="221" font-size="39" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="506.0" y="88" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="650">W<tspan baseline-shift="sub" font-size="70%">Q</tspan><tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="506.0" y="116" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">4 × 2</text><rect x="440" y="130" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="506" y="130" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="440" y="174" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="506" y="174" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="440" y="218" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="506" y="218" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="440" y="262" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="506" y="262" width="66" height="44" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><path d="M622 216 L745 216 M735.7893900599712 219.8941834230865 L745 216 L735.7893900599712 212.1058165769135" fill="none" stroke="#8b2cde" stroke-width="2.5"/><text x="900.0" y="51" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="650">Q<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="900.0" y="79" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="837" y="93" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="93" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="117" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="117" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="141" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="141" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="165" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="165" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="189" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="189" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="213" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="213" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="237" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="237" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="261" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="261" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="285" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="285" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="309" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="900" y="309" width="63" height="24" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="837" y="309" width="126" height="24" rx="4" fill="none" stroke="#8b2cde" stroke-width="2.5"/><text x="330" y="379" font-size="26" fill="#14171f" text-anchor="start" font-weight="500">Every token uses the same projection matrix within this head.</text></svg>

Ten input rows × four coordinates. Multiplying by a 4 × 2 projection gives ten query rows × two coordinates.



<a id="s04-v-grid"></a>
## One head makes one attention grid

[Matching slide](../../part3.html?present#s04/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 437" role="img" aria-label="One head: query-key scores, a causal weight grid, then weighted values" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>One head: query-key scores, a causal weight grid, then weighted values</title><text x="67.0" y="50" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="650">Q<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="67.0" y="78" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="36" y="92" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="92" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="114" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="114" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="136" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="136" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="158" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="158" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="180" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="180" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="202" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="202" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="224" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="224" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="246" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="246" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="268" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="268" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="290" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="67" y="290" width="31" height="22" rx="4" fill="#8b2cde16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="36" y="290" width="62" height="22" rx="4" fill="none" stroke="#8b2cde" stroke-width="2.5"/><text x="133" y="211" font-size="31" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="306.0" y="124" font-size="28" fill="#aa4e08" text-anchor="middle" font-weight="650">K<tspan baseline-shift="super" font-size="65%">(1)</tspan>ᵀ</text><text x="306.0" y="152" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">2 × 10</text><rect x="186" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="210" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="234" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="258" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="282" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="306" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="330" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="354" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="378" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="402" y="166" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="186" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="210" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="234" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="258" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="282" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="306" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="330" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="354" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="378" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><rect x="402" y="193" width="24" height="27" rx="4" fill="#aa4e0816" stroke="#d9dfe9" stroke-width="0.6"/><path d="M449 197 L514 197 M504.78939005997114 200.8941834230865 L514 197 L504.78939005997114 193.1058165769135" fill="none" stroke="#be123c" stroke-width="2.5"/><text x="490" y="279" font-size="22" fill="#be123c" text-anchor="middle" font-weight="500">÷ √2</text><text x="490" y="310" font-size="22" fill="#be123c" text-anchor="middle" font-weight="500">mask</text><text x="490" y="341" font-size="22" fill="#be123c" text-anchor="middle" font-weight="500">softmax</text><text x="675.0" y="50" font-size="28" fill="#be123c" text-anchor="middle" font-weight="650">A<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="675.0" y="78" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 10</text><rect x="555" y="92" width="24" height="22" rx="4" fill="#be123cb7" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="591.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="603" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="615.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="627" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="639.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="651" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="663.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="675" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="687.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="699" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="723" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="92" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="106.96000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="114" width="24" height="22" rx="4" fill="#be123c64" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="114" width="24" height="22" rx="4" fill="#be123c64" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="615.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="627" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="639.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="651" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="663.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="675" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="687.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="699" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="723" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="114" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="128.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="136" width="24" height="22" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="136" width="24" height="22" rx="4" fill="#be123ca1" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="136" width="24" height="22" rx="4" fill="#be123c1e" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="639.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="651" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="663.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="675" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="687.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="699" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="723" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="136" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="150.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="158" width="24" height="22" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="158" width="24" height="22" rx="4" fill="#be123c97" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="158" width="24" height="22" rx="4" fill="#be123c1c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="158" width="24" height="22" rx="4" fill="#be123c1f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="158" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="663.0" y="172.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="675" y="158" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="687.0" y="172.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="699" y="158" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.0" y="172.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="723" y="158" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="172.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="158" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="172.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="158" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="172.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="180" width="24" height="22" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="180" width="24" height="22" rx="4" fill="#be123c9a" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="180" width="24" height="22" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="180" width="24" height="22" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="180" width="24" height="22" rx="4" fill="#be123c18" stroke="#d9dfe9" stroke-width="0.6"/><rect x="675" y="180" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="687.0" y="194.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="699" y="180" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.0" y="194.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="723" y="180" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="194.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="180" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="194.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="180" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="194.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="202" width="24" height="22" rx="4" fill="#be123c2b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="202" width="24" height="22" rx="4" fill="#be123c2f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="202" width="24" height="22" rx="4" fill="#be123c2b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="202" width="24" height="22" rx="4" fill="#be123c2c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="202" width="24" height="22" rx="4" fill="#be123c2b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="675" y="202" width="24" height="22" rx="4" fill="#be123c31" stroke="#d9dfe9" stroke-width="0.6"/><rect x="699" y="202" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.0" y="216.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="723" y="202" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="216.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="202" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="216.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="202" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="216.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="224" width="24" height="22" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="224" width="24" height="22" rx="4" fill="#be123c34" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="224" width="24" height="22" rx="4" fill="#be123c1e" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="224" width="24" height="22" rx="4" fill="#be123c1f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="224" width="24" height="22" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="675" y="224" width="24" height="22" rx="4" fill="#be123c4b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="699" y="224" width="24" height="22" rx="4" fill="#be123c29" stroke="#d9dfe9" stroke-width="0.6"/><rect x="723" y="224" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="735.0" y="238.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="747" y="224" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="238.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="224" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="238.95999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="246" width="24" height="22" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="246" width="24" height="22" rx="4" fill="#be123c2f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="246" width="24" height="22" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="246" width="24" height="22" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="246" width="24" height="22" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="675" y="246" width="24" height="22" rx="4" fill="#be123c88" stroke="#d9dfe9" stroke-width="0.6"/><rect x="699" y="246" width="24" height="22" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="723" y="246" width="24" height="22" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="747" y="246" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="759.0" y="260.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="771" y="246" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="260.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="268" width="24" height="22" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="268" width="24" height="22" rx="4" fill="#be123c32" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="268" width="24" height="22" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="268" width="24" height="22" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="268" width="24" height="22" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="675" y="268" width="24" height="22" rx="4" fill="#be123c71" stroke="#d9dfe9" stroke-width="0.6"/><rect x="699" y="268" width="24" height="22" rx="4" fill="#be123c1f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="723" y="268" width="24" height="22" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="747" y="268" width="24" height="22" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="771" y="268" width="24" height="22" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="783.0" y="282.96" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="555" y="290" width="24" height="22" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="579" y="290" width="24" height="22" rx="4" fill="#be123c2d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="603" y="290" width="24" height="22" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="627" y="290" width="24" height="22" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="290" width="24" height="22" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="675" y="290" width="24" height="22" rx="4" fill="#be123c88" stroke="#d9dfe9" stroke-width="0.6"/><rect x="699" y="290" width="24" height="22" rx="4" fill="#be123c1a" stroke="#d9dfe9" stroke-width="0.6"/><rect x="723" y="290" width="24" height="22" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="747" y="290" width="24" height="22" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="771" y="290" width="24" height="22" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="555" y="290" width="240" height="22" rx="4" fill="none" stroke="#be123c" stroke-width="2.5"/><text x="821" y="207" font-size="31" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="894.0" y="50" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">V<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="894.0" y="78" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="863" y="92" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="92" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="114" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="114" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="136" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="136" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="158" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="158" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="180" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="180" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="202" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="202" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="224" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="224" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="246" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="246" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="268" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="268" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="863" y="290" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="894" y="290" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><path d="M943 197 L983 197 M973.7893900599712 200.8941834230865 L983 197 L973.7893900599712 193.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="1052.0" y="50" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">H<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="1052.0" y="78" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="1021" y="92" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="92" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="114" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="114" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="136" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="136" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="158" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="158" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="180" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="180" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="202" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="202" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="224" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="224" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="246" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="246" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="268" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="268" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="290" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1052" y="290" width="31" height="22" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="1021" y="290" width="62" height="22" rx="4" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="580" y="395" font-size="25" fill="#14171f" text-anchor="middle" font-weight="500">Each highlighted row follows receiver 10. Columns of A are source positions.</text></svg>

Keep Part II’s notation: \(M\) is the causal mask, \(A=\operatorname{softmax}(QK^\top/\sqrt{d_k}+M)\), and \(H=AV\) stores the message rows.



<a id="s04-v-parallel"></a>
## Two heads make two attention grids

[Matching slide](../../part3.html?present#s04/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 495" role="img" aria-label="Two independent attention grids computed from the same input snapshot" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Two independent attention grids computed from the same input snapshot</title><text x="109.0" y="96" font-size="28" fill="#245edb" text-anchor="middle" font-weight="650">same E</text><text x="109.0" y="124" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 4</text><rect x="51" y="138" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="138" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="138" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="138" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="159" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="159" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="159" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="159" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="180" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="180" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="180" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="180" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="201" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="201" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="201" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="201" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="222" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="222" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="222" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="222" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="243" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="243" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="243" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="243" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="264" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="264" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="264" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="264" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="285" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="285" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="285" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="285" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="306" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="306" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="306" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="306" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="327" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="80" y="327" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="109" y="327" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="138" y="327" width="29" height="21" rx="4" fill="#245edb16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="51" y="327" width="116" height="21" rx="4" fill="none" stroke="#245edb" stroke-width="2.5"/><path d="M167 243 H218 V156 H269" fill="none" stroke="#245edb" stroke-width="2.5"/><text x="283" y="66" font-size="28" fill="#8b2cde" text-anchor="start" font-weight="650">Head 1</text><rect x="270" y="93" width="194" height="94" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="367" y="132" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">own Q, K, V</text><text x="367" y="164" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">own projections</text><path d="M464 140 L532 140 M522.7893900599712 143.8941834230865 L532 140 L522.7893900599712 136.1058165769135" fill="none" stroke="#be123c" stroke-width="2.5"/><text x="644.0" y="44" font-size="28" fill="#be123c" text-anchor="middle" font-weight="650">A<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="644.0" y="72" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 10</text><rect x="569" y="86" width="15" height="14" rx="4" fill="#be123cb7" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="591.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="599" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="606.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="614" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="621.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="629" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="86" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="95.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="100" width="15" height="14" rx="4" fill="#be123c64" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="100" width="15" height="14" rx="4" fill="#be123c64" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="606.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="614" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="621.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="629" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="100" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="109.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="114" width="15" height="14" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="114" width="15" height="14" rx="4" fill="#be123ca1" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="114" width="15" height="14" rx="4" fill="#be123c1e" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="621.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="629" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="114" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="123.52000000000001" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="128" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="128" width="15" height="14" rx="4" fill="#be123c97" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="128" width="15" height="14" rx="4" fill="#be123c1c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="128" width="15" height="14" rx="4" fill="#be123c1f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="128" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="137.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="128" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="137.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="128" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="137.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="128" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="137.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="128" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="137.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="128" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="137.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="142" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="142" width="15" height="14" rx="4" fill="#be123c9a" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="142" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="142" width="15" height="14" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="142" width="15" height="14" rx="4" fill="#be123c18" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="142" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="151.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="142" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="151.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="142" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="151.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="142" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="151.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="142" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="151.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="156" width="15" height="14" rx="4" fill="#be123c2b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="156" width="15" height="14" rx="4" fill="#be123c2f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="156" width="15" height="14" rx="4" fill="#be123c2b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="156" width="15" height="14" rx="4" fill="#be123c2c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="156" width="15" height="14" rx="4" fill="#be123c2b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="156" width="15" height="14" rx="4" fill="#be123c31" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="156" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="165.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="156" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="165.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="156" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="165.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="156" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="165.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="170" width="15" height="14" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="170" width="15" height="14" rx="4" fill="#be123c34" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="170" width="15" height="14" rx="4" fill="#be123c1e" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="170" width="15" height="14" rx="4" fill="#be123c1f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="170" width="15" height="14" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="170" width="15" height="14" rx="4" fill="#be123c4b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="170" width="15" height="14" rx="4" fill="#be123c29" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="170" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="179.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="170" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="179.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="170" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="179.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="184" width="15" height="14" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="184" width="15" height="14" rx="4" fill="#be123c2f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="184" width="15" height="14" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="184" width="15" height="14" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="184" width="15" height="14" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="184" width="15" height="14" rx="4" fill="#be123c88" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="184" width="15" height="14" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="184" width="15" height="14" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="689" y="184" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="193.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="184" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="193.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="198" width="15" height="14" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="198" width="15" height="14" rx="4" fill="#be123c32" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="198" width="15" height="14" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="198" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="198" width="15" height="14" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="198" width="15" height="14" rx="4" fill="#be123c71" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="198" width="15" height="14" rx="4" fill="#be123c1f" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="198" width="15" height="14" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="689" y="198" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="704" y="198" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="207.51999999999998" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="212" width="15" height="14" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="212" width="15" height="14" rx="4" fill="#be123c2d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="212" width="15" height="14" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="212" width="15" height="14" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="212" width="15" height="14" rx="4" fill="#be123c13" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="212" width="15" height="14" rx="4" fill="#be123c88" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="212" width="15" height="14" rx="4" fill="#be123c1a" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="212" width="15" height="14" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="689" y="212" width="15" height="14" rx="4" fill="#be123c14" stroke="#d9dfe9" stroke-width="0.6"/><rect x="704" y="212" width="15" height="14" rx="4" fill="#be123c12" stroke="#d9dfe9" stroke-width="0.6"/><rect x="569" y="212" width="150" height="14" rx="4" fill="none" stroke="#be123c" stroke-width="2.5"/><path d="M729 154 L800 154 M790.7893900599712 157.8941834230865 L800 154 L790.7893900599712 150.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="761" y="120" font-size="21" fill="#0f766e" text-anchor="middle" font-weight="500">× V</text><text x="898.0" y="44" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">H<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="898.0" y="72" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="858" y="86" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="86" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="100" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="100" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="114" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="114" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="128" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="128" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="142" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="142" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="156" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="156" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="170" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="170" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="184" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="184" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="198" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="198" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="212" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="212" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="212" width="80" height="14" rx="4" fill="none" stroke="#0f766e" stroke-width="2.5"/><path d="M167 243 H218 V390 H269" fill="none" stroke="#245edb" stroke-width="2.5"/><text x="283" y="300" font-size="28" fill="#8b2cde" text-anchor="start" font-weight="650">Head 2</text><rect x="270" y="327" width="194" height="94" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="367" y="366" font-size="24" fill="#8b2cde" text-anchor="middle" font-weight="500">own Q, K, V</text><text x="367" y="398" font-size="20" fill="#586174" text-anchor="middle" font-weight="500">own projections</text><path d="M464 374 L532 374 M522.7893900599712 377.89418342308653 L532 374 L522.7893900599712 370.10581657691347" fill="none" stroke="#be123c" stroke-width="2.5"/><text x="644.0" y="278" font-size="28" fill="#be123c" text-anchor="middle" font-weight="650">A<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="644.0" y="306" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 10</text><rect x="569" y="320" width="15" height="14" rx="4" fill="#be123cb7" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="591.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="599" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="606.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="614" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="621.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="629" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="320" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="329.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="334" width="15" height="14" rx="4" fill="#be123c64" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="334" width="15" height="14" rx="4" fill="#be123c64" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="606.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="614" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="621.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="629" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="334" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="343.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="348" width="15" height="14" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="348" width="15" height="14" rx="4" fill="#be123c9b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="348" width="15" height="14" rx="4" fill="#be123c24" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="621.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="629" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="348" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="357.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="362" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="362" width="15" height="14" rx="4" fill="#be123c97" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="362" width="15" height="14" rx="4" fill="#be123c21" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="362" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="362" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="636.5" y="371.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="644" y="362" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="371.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="362" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="371.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="362" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="371.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="362" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="371.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="362" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="371.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="376" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="376" width="15" height="14" rx="4" fill="#be123c9c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="376" width="15" height="14" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="376" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="376" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="376" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="651.5" y="385.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="659" y="376" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="385.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="376" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="385.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="376" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="385.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="376" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="385.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="390" width="15" height="14" rx="4" fill="#be123c2c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="390" width="15" height="14" rx="4" fill="#be123c30" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="390" width="15" height="14" rx="4" fill="#be123c2d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="390" width="15" height="14" rx="4" fill="#be123c2c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="390" width="15" height="14" rx="4" fill="#be123c2c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="390" width="15" height="14" rx="4" fill="#be123c2c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="390" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="666.5" y="399.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="674" y="390" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="399.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="390" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="399.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="390" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="399.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="404" width="15" height="14" rx="4" fill="#be123c23" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="404" width="15" height="14" rx="4" fill="#be123c48" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="404" width="15" height="14" rx="4" fill="#be123c29" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="404" width="15" height="14" rx="4" fill="#be123c23" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="404" width="15" height="14" rx="4" fill="#be123c22" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="404" width="15" height="14" rx="4" fill="#be123c22" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="404" width="15" height="14" rx="4" fill="#be123c23" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="404" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="681.5" y="413.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="689" y="404" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="413.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="404" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="413.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="418" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="418" width="15" height="14" rx="4" fill="#be123c8c" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="418" width="15" height="14" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="418" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="418" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="418" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="418" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="418" width="15" height="14" rx="4" fill="#be123c17" stroke="#d9dfe9" stroke-width="0.6"/><rect x="689" y="418" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="696.5" y="427.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="704" y="418" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="427.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="432" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="432" width="15" height="14" rx="4" fill="#be123c6a" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="432" width="15" height="14" rx="4" fill="#be123c20" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="432" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="432" width="15" height="14" rx="4" fill="#be123c18" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="432" width="15" height="14" rx="4" fill="#be123c18" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="432" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="432" width="15" height="14" rx="4" fill="#be123c19" stroke="#d9dfe9" stroke-width="0.6"/><rect x="689" y="432" width="15" height="14" rx="4" fill="#be123c22" stroke="#d9dfe9" stroke-width="0.6"/><rect x="704" y="432" width="15" height="14" rx="4" fill="#9da3ae33" stroke="#d9dfe9" stroke-width="0.6"/><text x="711.5" y="441.52" font-size="14" fill="#586174" text-anchor="middle" font-weight="500">×</text><rect x="569" y="446" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="584" y="446" width="15" height="14" rx="4" fill="#be123c84" stroke="#d9dfe9" stroke-width="0.6"/><rect x="599" y="446" width="15" height="14" rx="4" fill="#be123c1b" stroke="#d9dfe9" stroke-width="0.6"/><rect x="614" y="446" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="629" y="446" width="15" height="14" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="644" y="446" width="15" height="14" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="659" y="446" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="674" y="446" width="15" height="14" rx="4" fill="#be123c16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="689" y="446" width="15" height="14" rx="4" fill="#be123c1d" stroke="#d9dfe9" stroke-width="0.6"/><rect x="704" y="446" width="15" height="14" rx="4" fill="#be123c15" stroke="#d9dfe9" stroke-width="0.6"/><rect x="569" y="446" width="150" height="14" rx="4" fill="none" stroke="#be123c" stroke-width="2.5"/><path d="M729 388 L800 388 M790.7893900599712 391.89418342308653 L800 388 L790.7893900599712 384.10581657691347" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="761" y="354" font-size="21" fill="#0f766e" text-anchor="middle" font-weight="500">× V</text><text x="898.0" y="278" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">H<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="898.0" y="306" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="858" y="320" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="320" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="334" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="334" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="348" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="348" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="362" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="362" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="376" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="376" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="390" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="390" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="404" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="404" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="418" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="418" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="432" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="432" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="446" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="898" y="446" width="40" height="14" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="858" y="446" width="80" height="14" rx="4" fill="none" stroke="#0f766e" stroke-width="2.5"/></svg>

Both heads receive the same E. Each uses its own projections, attention grid and values. Neither reads the other head’s output.



<a id="s04-v-joined"></a>
## Join coordinates, not token rows

[Matching slide](../../part3.html?present#s04/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 440" role="img" aria-label="Concatenate columns, project, and preserve the token axis" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Concatenate columns, project, and preserve the token axis</title><text x="61.0" y="66" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">H<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="61.0" y="94" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="27" y="108" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="108" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="131" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="131" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="154" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="154" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="177" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="177" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="200" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="200" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="223" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="223" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="246" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="246" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="269" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="269" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="292" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="292" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="315" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="61" y="315" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="27" y="315" width="68" height="23" rx="4" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="205.0" y="66" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">H<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="205.0" y="94" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 2</text><rect x="171" y="108" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="108" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="131" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="131" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="154" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="154" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="177" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="177" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="200" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="200" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="223" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="223" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="246" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="246" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="269" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="269" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="292" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="292" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="315" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="205" y="315" width="34" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="171" y="315" width="68" height="23" rx="4" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="144" y="232" font-size="32" fill="#586174" text-anchor="middle" font-weight="500">;</text><path d="M258 224 L320 224 M310.78939005997114 227.8941834230865 L320 224 L310.78939005997114 220.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><text x="423.0" y="66" font-size="28" fill="#0f766e" text-anchor="middle" font-weight="650">Concat(H<tspan baseline-shift="super" font-size="65%">(1)</tspan>, H<tspan baseline-shift="super" font-size="65%">(2)</tspan>)</text><text x="423.0" y="94" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 4</text><rect x="361" y="108" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="108" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="108" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="108" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="131" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="131" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="131" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="131" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="154" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="154" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="154" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="154" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="177" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="177" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="177" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="177" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="200" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="200" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="200" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="200" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="223" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="223" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="223" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="223" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="246" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="246" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="246" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="246" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="269" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="269" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="269" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="269" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="292" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="292" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="292" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="292" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="315" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="392" y="315" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="423" y="315" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="454" y="315" width="31" height="23" rx="4" fill="#0f766e16" stroke="#d9dfe9" stroke-width="0.6"/><rect x="361" y="315" width="124" height="23" rx="4" fill="none" stroke="#0f766e" stroke-width="2.5"/><path d="M423 108 V338" fill="none" stroke="#14171f" stroke-width="2.5"/><text x="529" y="233" font-size="34" fill="#14171f" text-anchor="start" font-weight="500">×</text><text x="651.0" y="115" font-size="28" fill="#147737" text-anchor="middle" font-weight="650">W<tspan baseline-shift="sub" font-size="70%">O</tspan></text><text x="651.0" y="143" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">4 × 4</text><rect x="589" y="157" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="620" y="157" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="157" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="682" y="157" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="589" y="189" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="620" y="189" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="189" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="682" y="189" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="589" y="221" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="620" y="221" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="221" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="682" y="221" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="589" y="253" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="620" y="253" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="651" y="253" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="682" y="253" width="31" height="32" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><path d="M745 225 L813 225 M803.7893900599712 228.8941834230865 L813 225 L803.7893900599712 221.1058165769135" fill="none" stroke="#147737" stroke-width="2.5"/><text x="917.0" y="66" font-size="28" fill="#147737" text-anchor="middle" font-weight="650">ΔE</text><text x="917.0" y="94" font-size="22" fill="#586174" text-anchor="middle" font-weight="500">10 × 4</text><rect x="855" y="108" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="108" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="108" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="108" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="131" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="131" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="131" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="131" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="154" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="154" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="154" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="154" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="177" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="177" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="177" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="177" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="200" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="200" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="200" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="200" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="223" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="223" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="223" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="223" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="246" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="246" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="246" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="246" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="269" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="269" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="269" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="269" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="292" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="292" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="292" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="292" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="315" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="886" y="315" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="917" y="315" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="948" y="315" width="31" height="23" rx="4" fill="#14773716" stroke="#d9dfe9" stroke-width="0.6"/><rect x="855" y="315" width="124" height="23" rx="4" fill="none" stroke="#147737" stroke-width="2.5"/><text x="580" y="398" font-size="28" fill="#14171f" text-anchor="middle" font-weight="500">10 message rows → 10 update rows. The token count stays fixed.</text></svg>

\(\Delta E=\operatorname{Concat}(H^{(1)},H^{(2)})W_O\), then \(E^{\prime}=E+\Delta E\). The superscript labels the head; each matrix still has ten token rows.



<a id="s05-v-scratch"></a>
## One head is still the Part II calculation

[Matching slide](../../part3.html?present#s05/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 100" role="img" aria-label="One head in code" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>One head in code</title><text x="580" y="55" font-size="31" fill="#8b2cde" text-anchor="middle" font-weight="500">E → Q, K, V → scores → weights → message</text></svg>

No new attention rule is needed. We reuse the same calculation with different learned matrices.

```python
def head(E, W_Q, W_K, W_V):
    Q, K, V = E @ W_Q, E @ W_K, E @ W_V
    scores = Q @ K.T / math.sqrt(Q.shape[-1])
    future = torch.ones(len(E), len(E), dtype=torch.bool).triu(1)
    A = scores.masked_fill(future, -torch.inf).softmax(-1)
    return A @ V
```



<a id="s05-v-combine"></a>
## Call it with two sets of parameters

[Matching slide](../../part3.html?present#s05/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 100" role="img" aria-label="Combine two head outputs in code" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Combine two head outputs in code</title><text x="580" y="54" font-size="31" fill="#0f766e" text-anchor="middle" font-weight="500">H<tspan baseline-shift="super" font-size="65%">(1)</tspan>, H<tspan baseline-shift="super" font-size="65%">(2)</tspan> → concatenate columns → W<tspan baseline-shift="sub" font-size="70%">O</tspan> → ΔE</text></svg>

This unbatched example keeps one row per token. The notebook checks these outputs against the printed numbers.

```python
H1 = head(E, W_Q1, W_K1, W_V1)
H2 = head(E, W_Q2, W_K2, W_V2)
joined = torch.cat([H1, H2], dim=-1)  # [10, 4]
delta_E = joined @ W_O
E_prime = E + delta_E
```



<a id="s05-v-train"></a>
## The same loss trains both heads

[Matching slide](../../part3.html?present#s05/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 455" role="img" aria-label="The same next-token path, with two parallel heads" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The same next-token path, with two parallel heads</title><rect x="23" y="136" width="177" height="69" rx="4" fill="#245edb08" stroke="#245edb" stroke-width="1.5"/><text x="111.5" y="164" font-size="25" fill="#245edb" text-anchor="middle" font-weight="600">Input E</text><text x="111.5" y="189" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 rows × 4</text><path d="M200 170 H238 V90 H285" fill="none" stroke="#245edb" stroke-width="2.5"/><rect x="285" y="55" width="235" height="69" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="83" font-size="25" fill="#8b2cde" text-anchor="middle" font-weight="600">Head 1</text><text x="402.5" y="108" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">Q, K, V → A → AV</text><path d="M520 90 L602 90 M592.7893900599712 93.8941834230865 L602 90 L592.7893900599712 86.1058165769135" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="604" y="55" width="182" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="695.0" y="83" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">H<tspan baseline-shift="super" font-size="65%">(1)</tspan></text><text x="695.0" y="108" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 messages × 2</text><path d="M786 90 H821 V149 H867" fill="none" stroke="#0f766e" stroke-width="2.5"/><path d="M200 170 H238 V264 H285" fill="none" stroke="#245edb" stroke-width="2.5"/><rect x="285" y="229" width="235" height="69" rx="4" fill="#8b2cde08" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="257" font-size="25" fill="#8b2cde" text-anchor="middle" font-weight="600">Head 2</text><text x="402.5" y="282" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">Q, K, V → A → AV</text><path d="M520 264 L602 264 M592.7893900599712 267.89418342308653 L602 264 L592.7893900599712 260.10581657691347" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="604" y="229" width="182" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="695.0" y="257" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">H<tspan baseline-shift="super" font-size="65%">(2)</tspan></text><text x="695.0" y="282" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">10 messages × 2</text><path d="M786 264 H821 V149 H867" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="867" y="115" width="265" height="69" rx="4" fill="#0f766e08" stroke="#0f766e" stroke-width="1.5"/><text x="999.5" y="143" font-size="25" fill="#0f766e" text-anchor="middle" font-weight="600">Concatenate</text><text x="999.5" y="168" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">two [10×2] → [10×4]</text><path d="M1000 184 L1000 222 M996.1058165769135 212.78939005997114 L1000 222 L1003.8941834230865 212.78939005997114" fill="none" stroke="#0f766e" stroke-width="2.5"/><rect x="867" y="226" width="265" height="69" rx="4" fill="#14773708" stroke="#147737" stroke-width="1.5"/><text x="999.5" y="254" font-size="25" fill="#147737" text-anchor="middle" font-weight="600">Project with W<tspan baseline-shift="sub" font-size="70%">O</tspan></text><text x="999.5" y="279" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">[10×4] × [4×4]</text><path d="M110 136 V40 H1148 V379 H1046" fill="none" stroke="#245edb" stroke-width="2" stroke-dasharray="7 6"/><path d="M1046 379 L1025 379 M1034.210609940029 375.10581657691347 L1025 379 L1034.210609940029 382.89418342308653" fill="none" stroke="#245edb" stroke-width="2.5"/><text x="980" y="25" font-size="20" fill="#245edb" text-anchor="middle" font-weight="500">keep original E</text><path d="M1000 295 L1000 353 M996.1058165769135 343.78939005997114 L1000 353 L1003.8941834230865 343.78939005997114" fill="none" stroke="#147737" stroke-width="2.5"/><text x="986" y="329" font-size="21" fill="#147737" text-anchor="end" font-weight="500">ΔE [10×4]</text><circle cx="1000" cy="379" r="24" fill="white" stroke="#147737" stroke-width="2"/><text x="1000" y="388" font-size="30" fill="#147737" text-anchor="middle" font-weight="500">+</text><path d="M973 379 L822 379 M831.2106099400288 375.10581657691347 L822 379 L831.2106099400288 382.89418342308653" fill="none" stroke="#147737" stroke-width="2.5"/><rect x="608" y="345" width="210" height="69" rx="4" fill="white" stroke="#d9dfe9" stroke-width="1.5"/><text x="713.0" y="373" font-size="25" fill="#586174" text-anchor="middle" font-weight="600">E′ = E + ΔE</text><text x="713.0" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">same 10 × 4 shape</text><path d="M608 380 L520 380 M529.2106099400288 376.10581657691347 L520 380 L529.2106099400288 383.89418342308653" fill="none" stroke="#147737" stroke-width="2.5"/><rect x="285" y="345" width="235" height="69" rx="4" fill="#245edb08" stroke="#245edb" stroke-width="1.5"/><text x="402.5" y="373" font-size="25" fill="#245edb" text-anchor="middle" font-weight="600">last row → MLP</text><text x="402.5" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">next-token prediction</text><path d="M285 380 L200 380 M209.21060994002886 376.10581657691347 L200 380 L209.21060994002886 383.89418342308653" fill="none" stroke="#be123c" stroke-width="2.5"/><rect x="23" y="345" width="177" height="69" rx="4" fill="#be123c08" stroke="#be123c" stroke-width="1.5"/><text x="111.5" y="373" font-size="25" fill="#be123c" text-anchor="middle" font-weight="600">Loss</text><text x="111.5" y="398" font-size="19" fill="#586174" text-anchor="middle" font-weight="500">observed target y</text></svg>

Next-token cross-entropy trains all the projections together. We do not label training examples “setting head” or “person head”.

<p>At training time, the observed next-token ID is the target for the vocabulary logits. At inference time, weights remain fixed: tokenize a prefix, run the model, choose a next token and append it. These steps are unchanged from Part II; the full executable loop remains in Notebook 7.</p>

<a id="s05-v-bias"></a>
## A bias adds a learned offset

[Matching slide](../../part3.html?present#s05/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 409" role="img" aria-label="A bias adds a learned offset after the projection; the displayed offset is illustrative" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>A bias adds a learned offset after the projection; the displayed offset is illustrative</title><text x="580" y="43" font-size="32" fill="#8b2cde" text-anchor="middle" font-weight="500">One head: qᵢ = eᵢ W<tspan baseline-shift="sub" font-size="70%">Q</tspan> + b<tspan baseline-shift="sub" font-size="70%">Q</tspan></text><text x="20" y="119" font-size="27" fill="#14171f" text-anchor="start" font-weight="650">bias=False</text><text x="340" y="119" font-size="24" fill="#8b2cde" text-anchor="start" font-weight="500">e₁₀ W<tspan baseline-shift="sub" font-size="70%">Q</tspan></text><rect x="653" y="83" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="725.5" y="115.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="798" y="83" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="870.5" y="115.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><text x="20" y="242" font-size="27" fill="#14171f" text-anchor="start" font-weight="650">bias=True</text><rect x="245" y="203" width="105" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="297.5" y="235.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><rect x="350" y="203" width="105" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="402.5" y="235.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.3</text><text x="479" y="239" font-size="32" fill="#14171f" text-anchor="start" font-weight="500">+</text><rect x="526" y="203" width="105" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="578.5" y="235.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">0.2</text><rect x="631" y="203" width="105" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="683.5" y="235.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">−0.1</text><text x="764" y="239" font-size="32" fill="#14171f" text-anchor="start" font-weight="500">=</text><rect x="815" y="203" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="887.5" y="235.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.5</text><rect x="960" y="203" width="145" height="50" rx="4" fill="#8b2cde0b" stroke="#8b2cde" stroke-width="1.5"/><text x="1032.5" y="235.5" font-size="26" fill="#8b2cde" text-anchor="middle" font-weight="500">2.2</text><text x="630" y="299" font-size="23" fill="#8b2cde" text-anchor="middle" font-weight="500">b<tspan baseline-shift="sub" font-size="70%">Q</tspan>: learned offset [2]</text><text x="580" y="369" font-size="29" fill="#14171f" text-anchor="middle" font-weight="500">The same b<tspan baseline-shift="sub" font-size="70%">Q</tspan> is added to every token row in this head.</text></svg>

The offset [0.2, −0.1] is illustrative. With bias=True, training learns it along with the projection weights. We use bias=False so the code matches the worksheet.

<p>A projection bias is shared across token positions (and batches). Position embeddings depend on the position, so these are different operations. A bias can shift the projection even for a zero input. Omitting it keeps this example simpler; it is not a general claim that bias-free attention is better.</p>

<a id="s05-v-bias-layers"></a>
## Which projections use the bias flag?

[Matching slide](../../part3.html?present#s05/5/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 429" role="img" aria-label="The bias flag controls all query, key, value and output projection biases, not the separate prediction MLP" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>The bias flag controls all query, key, value and output projection biases, not the separate prediction MLP</title><text x="23" y="40" font-size="29" fill="#14171f" text-anchor="start" font-weight="650">Inside nn.MultiheadAttention</text><text x="23" y="114" font-size="29" fill="#8b2cde" text-anchor="start" font-weight="500">Q = E W<tspan baseline-shift="sub" font-size="70%">Q</tspan> + b<tspan baseline-shift="sub" font-size="70%">Q</tspan></text><text x="23" y="156" font-size="22" fill="#586174" text-anchor="start" font-weight="500">offset after input projection</text><text x="401" y="114" font-size="29" fill="#aa4e08" text-anchor="start" font-weight="500">K = E W<tspan baseline-shift="sub" font-size="70%">K</tspan> + b<tspan baseline-shift="sub" font-size="70%">K</tspan></text><text x="401" y="156" font-size="22" fill="#586174" text-anchor="start" font-weight="500">offset after input projection</text><text x="779" y="114" font-size="29" fill="#0f766e" text-anchor="start" font-weight="500">V = E W<tspan baseline-shift="sub" font-size="70%">V</tspan> + b<tspan baseline-shift="sub" font-size="70%">V</tspan></text><text x="779" y="156" font-size="22" fill="#586174" text-anchor="start" font-weight="500">offset after input projection</text><text x="23" y="241" font-size="30" fill="#147737" text-anchor="start" font-weight="500">ΔE = Concat(H<tspan baseline-shift="super" font-size="65%">(1)</tspan>, H<tspan baseline-shift="super" font-size="65%">(2)</tspan>) W<tspan baseline-shift="sub" font-size="70%">O</tspan> + b<tspan baseline-shift="sub" font-size="70%">O</tspan></text><text x="23" y="282" font-size="23" fill="#586174" text-anchor="start" font-weight="500">offset after output projection</text><path d="M23 312 H1125" fill="none" stroke="#d9dfe9" stroke-width="2"/><text x="23" y="353" font-size="28" fill="#14171f" text-anchor="start" font-weight="500">Our worksheet and trained attention layers: bias=False.</text><text x="23" y="395" font-size="27" fill="#586174" text-anchor="start" font-weight="500">The separate prediction MLP still has its own biases.</text></svg>

PyTorch defaults to bias=True. In this lesson, bias=False removes the Q, K, V and output offsets. It leaves the mask, position embeddings and separate prediction layers unchanged.

<p>With total width four, bias=True adds four query offsets, four key offsets, four value offsets and four output offsets: 16 additional trainable scalars. Within each head the Q/K/V bias slice has width two. PyTorch stores the three input offsets together in in_proj_bias and the output offset in out_proj.bias. The distinct add_bias_kv option is not the bias flag discussed here. <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html">Official API documentation</a>. Both our scratch implementation and trained attention variants omit attention projection biases; their prediction MLP layers retain biases.</p>

In [5]:
# A learned offset is broadcast to each token row.
Q_no_bias = torch.tensor(case['heads'][0]['Q'])
b_Q = torch.tensor([0.2, -0.1])  # illustrative, not a fitted parameter
Q_with_bias = Q_no_bias + b_Q
print('Receiver 10:', Q_no_bias[-1].tolist(), '->', Q_with_bias[-1].tolist())
torch.testing.assert_close(Q_with_bias[-1], torch.tensor([2.5, 2.2]))

without_bias = nn.MultiheadAttention(4, 2, bias=False)
with_bias = nn.MultiheadAttention(4, 2, bias=True)
count = lambda layer: sum(p.numel() for p in layer.parameters())
print('Projection parameters:', count(without_bias), 'vs', count(with_bias))
print('Packed Q/K/V bias:', tuple(with_bias.in_proj_bias.shape))
print('Output bias:', tuple(with_bias.out_proj.bias.shape))
assert count(with_bias) - count(without_bias) == 16
assert without_bias.in_proj_bias is None
assert without_bias.out_proj.bias is None

Receiver 10: [2.299999952316284, 2.299999952316284] -> [2.5, 2.200000047683716]
Projection parameters: 64 vs 80
Packed Q/K/V bias: (12,)
Output bias: (4,)


<a id="s05-v-pytorch"></a>
## PyTorch packages the head calculation

[Matching slide](../../part3.html?present#s05/6/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 100" role="img" aria-label="PyTorch returns the projected update, before our residual" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>PyTorch returns the projected update, before our residual</title><text x="580" y="53" font-size="32" fill="#8b2cde" text-anchor="middle" font-weight="500">E → nn.MultiheadAttention → ΔE</text></svg>

Here E has shape [10, 4]: one unbatched sequence. PyTorch handles the projections and mixing. A contains two 10 × 10 weight grids.

```python
mha = nn.MultiheadAttention(embed_dim=4, num_heads=2, bias=False)
future = torch.ones(10, 10, dtype=torch.bool).triu(1)
delta_E, A = mha(E, E, E, attn_mask=future,
                 average_attn_weights=False)
E_prime = E + delta_E
```

<p>The library layer does not add position embeddings, the residual or the vocabulary classifier. attn_mask=future blocks future sources. average_attn_weights=False returns each head’s weight grid separately; it changes the returned diagnostic weights, not how the head messages combine. The notebook copies identical weights from our scratch implementation to PyTorch and checks both ΔE and the per-head attention weights. See the <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html">PyTorch documentation</a>.</p>

<a id="s06-v-width"></a>
## Keep the total width fixed

[Matching slide](../../part3.html?present#s06/1/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 430" role="img" aria-label="More heads divide a fixed total width into smaller per-head projections" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>More heads divide a fixed total width into smaller per-head projections</title><text x="22" y="108" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">1 head × 64 coordinates</text><rect x="460.0" y="74" width="640.0" height="62" rx="4" fill="#8b2cde0c" stroke="#8b2cde" stroke-width="2"/><text x="780.0" y="113" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="500">64</text><text x="22" y="259" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">4 heads × 16 coordinates</text><rect x="460.0" y="225" width="160.0" height="62" rx="4" fill="#8b2cde0c" stroke="#8b2cde" stroke-width="2"/><text x="540.0" y="264" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="500">16</text><rect x="620.0" y="225" width="160.0" height="62" rx="4" fill="#8b2cde0c" stroke="#8b2cde" stroke-width="2"/><text x="700.0" y="264" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="500">16</text><rect x="780.0" y="225" width="160.0" height="62" rx="4" fill="#8b2cde0c" stroke="#8b2cde" stroke-width="2"/><text x="860.0" y="264" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="500">16</text><rect x="940.0" y="225" width="160.0" height="62" rx="4" fill="#8b2cde0c" stroke="#8b2cde" stroke-width="2"/><text x="1020.0" y="264" font-size="28" fill="#8b2cde" text-anchor="middle" font-weight="500">16</text><text x="580" y="391" font-size="28" fill="#14171f" text-anchor="middle" font-weight="500">W<tspan baseline-shift="sub" font-size="70%">Q</tspan>, W<tspan baseline-shift="sub" font-size="70%">K</tspan>, W<tspan baseline-shift="sub" font-size="70%">V</tspan> and W<tspan baseline-shift="sub" font-size="70%">O</tspan> stay 64 × 64.</text></svg>

The trained comparison uses width 64. Four smaller heads give four attention patterns with the same attention parameter count as one wide head.

<p>Implementations often concatenate the per-head W_Q matrices into one large W_Q (and similarly for K and V). They project E first and reshape the projected coordinates into heads. They do not assign disjoint slices of raw E to different heads. Our two-head arithmetic toy uses width 4; this experiment uses width 64.</p>

<a id="s06-v-scores"></a>
## Compare held-out next-token prediction

[Matching slide](../../part3.html?present#s06/2/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 327" role="img" aria-label="Model; Cross-entropy ↓; Perplexity ↓" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Model; Cross-entropy ↓; Perplexity ↓</title><text x="37" y="47" font-size="25" fill="#586174" text-anchor="start" font-weight="650">Model</text><text x="446" y="47" font-size="25" fill="#586174" text-anchor="start" font-weight="650">Cross-entropy ↓</text><text x="813" y="47" font-size="25" fill="#586174" text-anchor="start" font-weight="650">Perplexity ↓</text><text x="37" y="116" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">Embedding → MLP</text><text x="446" y="116" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">3.943</text><text x="813" y="116" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">51.59</text><path d="M25 134 H1120" fill="none" stroke="#d9dfe9" stroke-width="1.5"/><text x="37" y="187" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">One attention head</text><text x="446" y="187" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">3.445</text><text x="813" y="187" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">31.34</text><path d="M25 205 H1120" fill="none" stroke="#d9dfe9" stroke-width="1.5"/><text x="37" y="258" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">Four attention heads</text><text x="446" y="258" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">3.360</text><text x="813" y="258" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">28.78</text><path d="M25 276 H1120" fill="none" stroke="#d9dfe9" stroke-width="1.5"/></svg>

For each run, perplexity = exp(cross-entropy using natural logs). Lower means higher probability for observed tokens. The table averages three seeds on the same test stories.

<p>Same 6,000-document TinyStories subset, tokenizer, 64-token context, 6,000-update budget and validation selection. Seeds 11, 29 and 47. More heads helped this experiment; this is not a guarantee for every dataset, head count or generated continuation. <a href="../../notebooks/wordlm/06_multihead_comparison.html">Full experiment and variation across seeds</a>.</p>

<a id="s06-v-cost"></a>
## Compare size and training time too

[Matching slide](../../part3.html?present#s06/3/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 327" role="img" aria-label="Model; Parameters; Training / seed" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Model; Parameters; Training / seed</title><text x="37" y="47" font-size="25" fill="#586174" text-anchor="start" font-weight="650">Model</text><text x="446" y="47" font-size="25" fill="#586174" text-anchor="start" font-weight="650">Parameters</text><text x="813" y="47" font-size="25" fill="#586174" text-anchor="start" font-weight="650">Training / seed</text><text x="37" y="116" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">Embedding → MLP</text><text x="446" y="116" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">2,332,832</text><text x="813" y="116" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">58.2 s</text><path d="M25 134 H1120" fill="none" stroke="#d9dfe9" stroke-width="1.5"/><text x="37" y="187" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">One attention head</text><text x="446" y="187" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">1,321,120</text><text x="813" y="187" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">65.6 s</text><path d="M25 205 H1120" fill="none" stroke="#d9dfe9" stroke-width="1.5"/><text x="37" y="258" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">Four attention heads</text><text x="446" y="258" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">1,321,120</text><text x="813" y="258" font-size="29" fill="#14171f" text-anchor="start" font-weight="500">69.7 s</text><path d="M25 276 H1120" fill="none" stroke="#d9dfe9" stroke-width="1.5"/></svg>

Mean training times on Apple M2 Max/MPS, including validation. The two attention models have equal parameter counts; four heads took slightly longer.



<a id="s06-v-demo"></a>
## Give all three models the same prompt

[Matching slide](../../part3.html?present#s06/4/0)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 212" role="img" aria-label="Compare real language models in the browser" style="font-family:Avenir Next,Segoe UI,Arial,sans-serif"><title>Compare real language models in the browser</title><text x="580" y="75" font-size="29" fill="#14171f" text-anchor="middle" font-weight="500">Training prefix → held-out story → a different kind of prompt</text><text x="580" y="155" font-size="27" fill="#586174" text-anchor="middle" font-weight="500">Compare the continuation and generation time on your device.</text></svg>

<a href="../../word-lab/" target="_blank" rel="noopener">Open the live models ↗</a> · <a href="../../notebooks/wordlm/07_multihead_step_by_step.html">Work through the code</a> · <a href="../../notebooks/wordlm/06_multihead_comparison.html">Inspect the trained experiment</a>

<p>The demo runs actual checkpoints with WebGPU or WebAssembly. It includes training, held-out and outside-domain prompts and measures generation locally. Normalization, full Transformer blocks and complexity remain in the <a href="../../part2b.html">optional Part 2B reference</a>. Formula source: <a href="https://arxiv.org/abs/1706.03762">Attention Is All You Need, §3.2.2</a>. Visual teaching references: <a href="https://www.3blue1brown.com/lessons/attention/">3Blue1Brown</a> and <a href="https://jalammar.github.io/illustrated-transformer/">Jay Alammar</a>.

<a id="executable-lab"></a>
# The executable lab

The lecture has now shown the whole idea. This optional lab slows down the code:
we create two examples, project the embeddings, expose the head axis, calculate
the messages and check the results. Here `D` means the same model width as
`d_model` in the lecture. The head axis has size 2; it is not the message matrix H.

The expanded figures below accompany the code rather than add new lecture slides.

<a id="s01-name"></a>
## Which earlier characters might help?

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="a  b  i → Embeddings → Read context → Next character" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>a  b  i → Embeddings → Read context → Next character</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">a  b  i</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">3 known IDs</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Embeddings</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">3 learned rows</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Read context</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">MLP or attention</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Next character</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">for example, d</text></g></svg>

Part I predicted the next character in a name such as aabid. The question stays the same; we are changing how the model reads the known context.

<a id="s01-river"></a>
## Back to the river bank

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 370" role="img" aria-label="The river sentence from Part II" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>The river sentence from Part II</title><g><rect x="20" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="60.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">The</text><text x="60.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">1</text></g><g><rect x="112" y="55" width="150" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="187.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">fisherman</text><text x="187.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">2</text></g><g><rect x="274" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="314.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">sat</text><text x="314.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">3</text></g><g><rect x="366" y="55" width="110" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="421.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">beside</text><text x="421.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4</text></g><g><rect x="488" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="528.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">the</text><text x="528.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">5</text></g><g><rect x="580" y="55" width="110" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="635.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">river</text><text x="635.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">6</text></g><g><rect x="702" y="55" width="110" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="757.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">bank</text><text x="757.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">7</text></g><g><rect x="824" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="864.0" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">and</text><text x="864.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">8</text></g><g><rect x="916" y="55" width="135" height="88" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="983.5" y="87" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">watched</text><text x="983.5" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">9</text></g><g><rect x="1063" y="55" width="80" height="88" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="1103.0" y="87" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">the</text><text x="1103.0" y="116" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">10</text></g><text x="580" y="325" fill="var(--ink)" font-size="30" text-anchor="middle" font-weight="500">Predict the next token after the final “the”.</text><text x="635.0" y="245" fill="var(--c-q)" font-size="30" text-anchor="middle" font-weight="500">Which setting?</text><text x="187.0" y="245" fill="var(--c-q)" font-size="30" text-anchor="middle" font-weight="500">Who is in the scene?</text><path d="M635.0 220 L635.0 145 M631.1058165769135 154.21060994002886 L635.0 145 L638.8941834230865 154.21060994002886" fill="none" stroke="var(--c-q)" stroke-width="2.5"/><path d="M187.0 220 L187.0 145 M183.1058165769135 154.21060994002886 L187.0 145 L190.8941834230865 154.21060994002886" fill="none" stroke="var(--c-q)" stroke-width="2.5"/></svg>

One query can benefit from several clues at once. What setting are we in? Who is there?

<a id="s01-scope"></a>
## Keep the two examples separate

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Worksheet; Trained experiment" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Worksheet; Trained experiment</title><g data-cell-left="20" data-cell-right="580"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Worksheet</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Trained experiment</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2 heads × 2 coordinates</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4 heads × 16 coordinates</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Hand-chosen projections</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Learned from TinyStories</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Part II token + position rows</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Same 64-token benchmark window</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The worksheet explains the arithmetic. Held-out results later tell us whether the trained models improved.

<a id="s02-break"></a>
## From one weight row to two

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

What changes when the same input passes through two sets of projections?

<a id="s02-one"></a>
## One head produces one message

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Full E → Q, K, V → Weights A → Message AV" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Full E → Q, K, V → Weights A → Message AV</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Full E</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">all coordinates</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q, K, V</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">learned projections</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Weights A</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one row per query</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Message AV</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one weighted sum</text></g></svg>

One head can already read several sources. Its value coordinates share the same attention weights.

<a id="s02-two"></a>
## Two heads keep two messages

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="397.5" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="192" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q¹ · K¹ · V¹</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="312" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q² · K² · V²</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

Each head has its own Q, K and V projections and its own softmax. Both receive the full E.

<a id="s02-full"></a>
## Project first; split the projected coordinates

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="E [B,T,4] → W_Q [4,4] → Q [B,T,4] → 2 heads × 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>E [B,T,4] → W_Q [4,4] → Q [B,T,4] → 2 heads × 2</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E [B,T,4]</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">full input rows</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">W_Q [4,4]</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">learned mixing</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q [B,T,4]</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">projected coordinates</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">2 heads × 2</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one view per head</text></g></svg>

We split Q, K and V after projection. We do not give head 1 the first half of the raw embedding and head 2 the second half.

<a id="s02-columns"></a>
## Separate heads can use different input features

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Input coordinate; W_Q: head 1; W_Q: head 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Input coordinate; W_Q: head 1; W_Q: head 2</title><g data-cell-left="20" data-cell-right="370"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Input coordinate</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">W_Q: head 1</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">W_Q: head 2</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">water</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">finance</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">person</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0]</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="370"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">glue</text></g><g data-cell-left="370" data-cell-right="755"><text x="380" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1, 1]</text></g><g data-cell-left="755" data-cell-right="1140"><text x="765" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1, 0]</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

These are two 4×2 matrices placed side by side. Every head can learn from every input coordinate.

<a id="s03-break"></a>
## Two heads, one receiving token

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Keep the final “the” at position 10 as the query throughout.

<a id="s03-input"></a>
## Start with the same position-aware rows

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Position / token; water; finance; person; glue" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Position / token; water; finance; person; glue</title><g data-cell-left="20" data-cell-right="350"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Position / token</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">water</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">finance</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">person</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">glue</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="350"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2 fisherman</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.0</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.1</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.1</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="350"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">6 river</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.1</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">−0.1</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.1</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="350"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">10 the</text></g><g data-cell-left="350" data-cell-right="545"><text x="360" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="545" data-cell-right="740"><text x="555" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="740" data-cell-right="935"><text x="750" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">0.0</text></g><g data-cell-left="935" data-cell-right="1140"><text x="945" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.3</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

E already includes token embedding + position embedding. Token IDs are not embedding coordinates.

<a id="s03-q1"></a>
## The first query asks about the setting

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="e₁₀ → W_Q¹ → q₁₀¹" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>e₁₀ → W_Q¹ → q₁₀¹</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">e₁₀</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[0.000, 0.000, 0.000, 2.300]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">W_Q¹</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 × 2</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">q₁₀¹</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[2.300, 2.300]</text></g></svg>

Our chosen projection copies the glue coordinate into both query coordinates: [2.3, 2.3].

<a id="s03-q2"></a>
## The second query uses a different projection

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="e₁₀ → W_Q² → q₁₀²" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>e₁₀ → W_Q² → q₁₀²</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">e₁₀</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[0.000, 0.000, 0.000, 2.300]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">W_Q²</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 × 2</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">q₁₀²</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[2.300, 0.000]</text></g></svg>

The same input row now produces [2.3, 0.0]. The query changes because the projection matrix changes.

<a id="s03-keys"></a>
## The source keys are different too

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Source; Key in head 1; Key in head 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Source; Key in head 1; Key in head 2</title><g data-cell-left="20" data-cell-right="360"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Source</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Key in head 1</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Key in head 2</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">fisherman</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.000, 0.100]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.100, 0.000]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">river</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.100, −0.100]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 0.100]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">bank</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.700, 0.700]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.100, 0.800]</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="360"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">the</text></g><g data-cell-left="360" data-cell-right="750"><text x="370" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 0.000]</text></g><g data-cell-left="750" data-cell-right="1140"><text x="760" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 2.300]</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

Head 1 exposes water/finance features. Head 2 exposes person/glue features. These labels belong to this designed worksheet.

<a id="s03-dot1"></a>
## Head 1 scores the river key

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Calculation; Value" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Calculation; Value</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Calculation</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">q₁₀¹</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.3, 2.3]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">k₆¹ for river</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.1, −0.1]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Dot product</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.3 × 3.1 + 2.3 × (−0.1) = 6.9</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Divide by √2</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4.879</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

The scaling uses the head width: two coordinates, not four.

<a id="s03-dot2"></a>
## Head 2 scores the fisherman key

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Calculation; Value" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Calculation; Value</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Calculation</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">q₁₀²</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.3, 0.0]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">k₂² for fisherman</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.1, 0.0]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Dot product</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2.3 × 2.1 + 0.0 × 0.0 = 4.83</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Divide by √2</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.415</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

We compare each head’s query only with keys from that same head.

<a id="s03-weights1"></a>
## Head 1: scores become a weight row

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 1" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 1</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 1 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><text x="165" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="120" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0838525915956979"/><text x="165" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.006</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><text x="266" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">3.42</text><rect x="221" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.17962740297689267"/><text x="266" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.166</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><text x="367" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.49</text><rect x="322" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08533354000056549"/><text x="367" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.009</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><text x="468" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.81</text><rect x="423" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08738376966024583"/><text x="468" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.012</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="569" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.33</text><rect x="524" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08453298482034709"/><text x="569" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.008</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><text x="670" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">4.88</text><rect x="625" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.5105865263773078"/><text x="670" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.718</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><text x="771" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">2.28</text><rect x="726" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11191242203024018"/><text x="771" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.053</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><text x="872" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="827" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="872" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><text x="973" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">1.14</text><rect x="928" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0902221140911639"/><text x="973" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.017</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="1074" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="1029" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="1074" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="15" y="137" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">score</text><text x="15" y="188" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

Each weight is exp(score) divided by the sum of exp(scores) in this row. All ten sources are allowed for the final query.

<a id="s03-weights2"></a>
## Head 2 has its own softmax

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 2" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 2</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 2 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><text x="165" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="120" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="165" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><text x="266" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">3.42</text><rect x="221" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.4957763940137875"/><text x="266" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.693</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><text x="367" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.98</text><rect x="322" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11625688180810867"/><text x="367" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.060</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><text x="468" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="423" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="468" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="569" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="524" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09366478175594563"/><text x="569" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.023</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><text x="670" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="625" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09366478175594563"/><text x="670" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.023</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><text x="771" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="726" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="771" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><text x="872" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="827" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09607807283342128"/><text x="872" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.027</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><text x="973" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">1.14</text><rect x="928" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.12266008757658185"/><text x="973" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.071</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="1074" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="1029" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.09366478175594563"/><text x="1074" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.023</text><text x="15" y="137" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">score</text><text x="15" y="188" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

Normalize over sources within head 2. Do not normalize across heads or average their scores.

<a id="s03-value1"></a>
## Head 1 sends water and finance information

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Source; Weight; Value row; Weighted contribution" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Source; Weight; Value row; Weighted contribution</title><g data-cell-left="20" data-cell-right="255"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Source</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weight</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value row</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weighted contribution</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">river</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="101" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500">0.718</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[3.100, −0.100]</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[2.225, −0.072]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">All other sources</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="152" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[0.392, 0.054]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Total message</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="203" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[2.617, −0.018]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The river contribution is its weight × its value row. Add the contributions from every source to get the two-coordinate message.

<a id="s03-value2"></a>
## Head 2 sends a different kind of message

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Source; Weight; Value row; Weighted contribution" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Source; Weight; Value row; Weighted contribution</title><g data-cell-left="20" data-cell-right="255"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Source</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weight</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Value row</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Weighted contribution</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">fisherman</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="101" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500">0.693</text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[2.100, 0.000]</text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="101" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[1.455, 0.000]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">All other sources</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="152" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="152" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[0.097, 0.538]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="255"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Total message</text></g><g data-cell-left="255" data-cell-right="400"><text x="265" y="203" fill="var(--c-a)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="400" data-cell-right="730"><text x="410" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500"></text></g><g data-cell-left="730" data-cell-right="1140"><text x="740" y="203" fill="var(--c-v)" font-size="26" text-anchor="start" font-weight="500">[1.552, 0.538]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The second head mixes its own values with its own weights. Matching features choose where to read; values carry the information.

<a id="s04-break"></a>
## Bring the two messages back together

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

How do two small message rows become one update to the original row?

<a id="s04-join"></a>
## Concatenate the messages, not the tokens

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Head; Message coordinates; Message" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Head; Message coordinates; Message</title><g data-cell-left="20" data-cell-right="180"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Head</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Message coordinates</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Message</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="180"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">water / finance</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.617, −0.018]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="180"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">person / glue</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1.552, 0.538]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="180"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Joined</text></g><g data-cell-left="180" data-cell-right="570"><text x="190" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">head 1, then head 2</text></g><g data-cell-left="570" data-cell-right="1140"><text x="580" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2.617, −0.018, 1.552, 0.538]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

Two 2-coordinate rows become one 4-coordinate row. Concatenation preserves both feature groups; it does not average them.

<a id="s04-wo"></a>
## W_O can mix information across heads

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Joined coordinate; Output weights" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Joined coordinate; Output weights</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Joined coordinate</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Output weights</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[1, 0, 0, 0]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 1, 0, 0]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.25, 0, 1, 0]</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4</text></g><g data-cell-left="420" data-cell-right="1140"><text x="430" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 0, 0, 1]</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

Water update = 2.617 + 0.25 × 1.552 = 3.005. W_O maps the joined message back to model width.

<a id="s04-residual"></a>
## Add the update to the original row

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Row; Four coordinates" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Row; Four coordinates</title><g data-cell-left="20" data-cell-right="470"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Row</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Four coordinates</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Original e₁₀</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0.000, 0.000, 0.000, 2.300]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Attention update Δe₁₀</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.005, −0.018, 1.552, 0.538]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Updated e′₁₀</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[3.005, −0.018, 1.552, 2.838]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The residual is the same addition as in Part II. There is still one updated row per input token.

<a id="s04-predict"></a>
## The prediction MLP stays in place

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="e′₁₀ → Hidden + ReLU → 20 logits → Softmax" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>e′₁₀ → Hidden + ReLU → 20 logits → Softmax</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">e′₁₀</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 coordinates</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Hidden + ReLU</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">8 activations</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">20 logits</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one per vocabulary item</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Softmax</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token probabilities</text></g></svg>

Choose a token only after vocabulary softmax. Attention weights choose source positions; vocabulary probabilities choose possible next tokens.

<a id="s04-live"></a>
## Change the context; inspect both heads

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 1" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 1</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 1 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><text x="165" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.16</text><rect x="120" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0838525915956979"/><text x="165" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.006</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><text x="266" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">3.42</text><rect x="221" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.17962740297689267"/><text x="266" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.166</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><text x="367" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.49</text><rect x="322" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08533354000056549"/><text x="367" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.009</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><text x="468" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.81</text><rect x="423" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08738376966024583"/><text x="468" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.012</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="569" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.33</text><rect x="524" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08453298482034709"/><text x="569" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.008</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><text x="670" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">4.88</text><rect x="625" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.5105865263773078"/><text x="670" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.718</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><text x="771" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">2.28</text><rect x="726" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11191242203024018"/><text x="771" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.053</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><text x="872" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="827" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="872" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><text x="973" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">1.14</text><rect x="928" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0902221140911639"/><text x="973" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.017</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><text x="1074" y="137" fill="var(--c-k)" font-size="23" text-anchor="middle" font-weight="500">0.00</text><rect x="1029" y="154" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="1074" y="188" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="15" y="137" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">score</text><text x="15" y="188" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

The matching slide lets you switch contexts and inspect either head. Here, compute both sentences and print their final-query messages. These are hand-chosen worksheet parameters, not trained results.

In [6]:
for name in ['river', 'cheque']:
    ids = torch.tensor([[word_to_id[w.lower()] for w in worksheet['sentences'][name]]])
    demo = load_worksheet_weights(TinyMultiHeadLM(len(word_to_id)), worksheet)
    E_demo = demo.token_embedding(ids) + demo.position_embedding(torch.arange(10))
    with torch.no_grad():
        _, weights = demo.attention(E_demo)
        values = demo.attention.split_heads(demo.attention.W_V(E_demo))
        messages = weights @ values
    print(name, 'final messages by head:', messages[0, :, -1].tolist())

river final messages by head: [[2.6168477535247803, -0.017858127132058144], [1.5519635677337646, 0.5383409261703491]]
cheque final messages by head: [[0.05300545319914818, 2.471384286880493], [2.3669915199279785, 0.4609062373638153]]


<a id="s05-break"></a>
## From the drawing to tensors

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Keep B = 2 examples, T = 10 tokens, D = 4 coordinates and n_heads = 2.

<a id="s05-model"></a>
## Create the tables, projections and prediction MLP

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Token + position → ScratchMultiHead → Hidden → vocabulary" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Token + position → ScratchMultiHead → Hidden → vocabulary</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token + position</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">20 × 4 and 10 × 4</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">ScratchMultiHead</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 coordinates, 2 heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Hidden → vocabulary</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">4 → 8 → 20</text></g></svg>

The download defines every layer. Load the hand-chosen worksheet weights to reproduce the printed numbers; ordinary training starts from random parameters.

### The complete implementation

This is the same source file imported above. Read the small snippets that follow alongside the diagram, then return here to see how they fit together. `load_worksheet_weights` copies the printed parameters so our outputs match the figures. It is not part of an ordinary training loop.

In [7]:
"""Part III: explicit multi-head self-attention and a small next-token model."""
import math
import torch
from torch import nn
from torch.nn import functional as F


class ScratchMultiHead(nn.Module):
    def __init__(self, width=4, heads=2):
        super().__init__()
        if heads < 1 or width % heads:
            raise ValueError('width must be divisible by a positive head count')
        self.width, self.heads = width, heads
        self.head_width = width // heads
        self.W_Q = nn.Linear(width, width, bias=False)
        self.W_K = nn.Linear(width, width, bias=False)
        self.W_V = nn.Linear(width, width, bias=False)
        self.W_O = nn.Linear(width, width, bias=False)

    def split_heads(self, rows):
        B, T, _ = rows.shape
        return rows.reshape(B, T, self.heads, self.head_width).transpose(1, 2)

    def forward(self, E, padding_mask=None):
        B, T, _ = E.shape
        Q = self.split_heads(self.W_Q(E))
        K = self.split_heads(self.W_K(E))
        V = self.split_heads(self.W_V(E))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_width)
        blocked = torch.ones(T, T, dtype=torch.bool, device=E.device).triu(1)
        if padding_mask is not None:
            # Ignore PAD sources for real queries. PAD-query outputs are unused;
            # leave their causal prefix available to avoid an all-masked softmax.
            blocked = blocked | (padding_mask[:, None, :] & ~padding_mask[:, :, None])
            blocked = blocked[:, None]
        weights = scores.masked_fill(blocked, float('-inf')).softmax(dim=-1)
        messages = weights @ V
        joined = messages.transpose(1, 2).contiguous().reshape(B, T, self.width)
        delta = self.W_O(joined)
        return delta, weights


class TinyMultiHeadLM(nn.Module):
    def __init__(self, vocab_size, context=10, width=4, heads=2, hidden=8):
        super().__init__()
        self.context = context
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context, width)
        self.attention = ScratchMultiHead(width, heads)
        self.hidden = nn.Linear(width, hidden)
        self.readout = nn.Linear(hidden, vocab_size)

    def forward(self, ids):
        T = ids.shape[1]
        if T > self.context:
            raise ValueError('Crop the prompt to the configured context window')
        E = self.token_embedding(ids) + self.position_embedding(torch.arange(T, device=ids.device))
        delta, _ = self.attention(E)
        updated = E + delta
        return self.readout(F.relu(self.hidden(updated[:, -1])))


def copy_to_pytorch(scratch):
    """Use exactly the same parameters, not a newly randomized comparison."""
    layer = nn.MultiheadAttention(scratch.width, scratch.heads, bias=False,
                                  dropout=0.0, batch_first=True)
    layer = layer.to(device=scratch.W_Q.weight.device, dtype=scratch.W_Q.weight.dtype)
    with torch.no_grad():
        layer.in_proj_weight.copy_(torch.cat([scratch.W_Q.weight,
                                             scratch.W_K.weight,
                                             scratch.W_V.weight], dim=0))
        layer.out_proj.weight.copy_(scratch.W_O.weight)
    return layer


def load_worksheet_weights(model, worksheet):
    """Load the printed, hand-chosen example; this is not a training algorithm."""
    heads = worksheet['headsLesson']['projections']
    with torch.no_grad():
        model.token_embedding.weight.copy_(torch.tensor([worksheet['tok_emb'][w] for w in worksheet['vocab']]))
        model.position_embedding.weight.copy_(torch.tensor(worksheet['pos_emb'][:model.context]))
        for letter in ['Q', 'K', 'V']:
            packed = [a + b for a, b in zip(heads[0][letter], heads[1][letter])]
            getattr(model.attention, 'W_' + letter).weight.copy_(torch.tensor(packed).T)
        model.attention.W_O.weight.copy_(torch.tensor(worksheet['headsLesson']['W_O']).T)
        model.hidden.weight.copy_(torch.tensor(worksheet['W_hidden']).T)
        model.hidden.bias.copy_(torch.tensor(worksheet['b_hidden']))
        model.readout.weight.copy_(torch.tensor(worksheet['W_vocab']).T)
        model.readout.bias.copy_(torch.tensor(worksheet['b_vocab']))
    return model


In [8]:
model = TinyMultiHeadLM(vocab_size=20, context=10,
                        width=4, heads=2, hidden=8)
load_worksheet_weights(model, worksheet)

TinyMultiHeadLM(
  (token_embedding): Embedding(20, 4)
  (position_embedding): Embedding(10, 4)
  (attention): ScratchMultiHead(
    (W_Q): Linear(in_features=4, out_features=4, bias=False)
    (W_K): Linear(in_features=4, out_features=4, bias=False)
    (W_V): Linear(in_features=4, out_features=4, bias=False)
    (W_O): Linear(in_features=4, out_features=4, bias=False)
  )
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (readout): Linear(in_features=8, out_features=20, bias=True)
)

<a id="s05-batch"></a>
## Two input windows form one batch

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 185" role="img" aria-label="Example; Ten token IDs; Observed next token" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Example; Ten token IDs; Observed next token</title><g data-cell-left="20" data-cell-right="190"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Example</text></g><g data-cell-left="190" data-cell-right="890"><text x="200" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Ten token IDs</text></g><g data-cell-left="890" data-cell-right="1140"><text x="900" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Observed next token</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="190"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">river</text></g><g data-cell-left="190" data-cell-right="890"><text x="200" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[0, 1, 2, 3, 0, 4, 5, 6, 7, 0]</text></g><g data-cell-left="890" data-cell-right="1140"><text x="900" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">water</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="190"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">cheque</text></g><g data-cell-left="190" data-cell-right="890"><text x="200" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[8, 9, 0, 10, 11, 0, 5, 6, 7, 0]</text></g><g data-cell-left="890" data-cell-right="1140"><text x="900" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">teller</text></g><path d="M20 165 H1140" stroke="var(--line)"/></svg>

These are integer IDs, not embeddings. Each example has one observed target.

In [9]:
X = torch.tensor([river_ids, cheque_ids])
y = torch.tensor([word_to_id["water"], word_to_id["teller"]])

<a id="s05-embed-map"></a>
## Find the embedding lookup on the map

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="397.5" y="50" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="192" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q¹ · K¹ · V¹</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="312" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q² · K² · V²</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

We have token IDs. Next, look up their learned rows and add the position rows. Both heads will read the result.

<a id="s05-embed"></a>
## Look up token rows and add position rows

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Token IDs [2,10] → Token + position rows → E [2,10,4]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Token IDs [2,10] → Token + position rows → E [2,10,4]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token IDs [2,10]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">integers</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token + position rows</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">lookup, then add</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E [2,10,4]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">floating-point vectors</text></g></svg>

The embedding tables are learned parameters. E contains floating-point vectors with shape [2,10,4].

In [10]:
positions = torch.arange(X.shape[1])
E = model.token_embedding(X) + model.position_embedding(positions)

<a id="s05-project"></a>
## Project the full input three times

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="E [2,10,4] → Three matrices → Q, K, V" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>E [2,10,4] → Three matrices → Q, K, V</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E [2,10,4]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">same input</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Three matrices</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">each 4 × 4</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q, K, V</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">each [2,10,4]</text></g></svg>

Different learned matrices give matching queries, matching keys and transmitted values.

In [11]:
Q = model.attention.W_Q(E)
K = model.attention.W_K(E)
V = model.attention.W_V(E)

<a id="s05-split"></a>
## Make the head axis explicit

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="[2,10,4] → [2,10,2,2] → [2,2,10,2]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>[2,10,4] → [2,10,2,2] → [2,2,10,2]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">[2,10,4]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, D</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">[2,10,2,2]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, heads, d_head</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">[2,2,10,2]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, heads, T, d_head</text></g></svg>

reshape groups projected coordinates; transpose puts heads before tokens. Apply the same operation to K and V.

In [12]:
Q = Q.reshape(2, 10, 2, 2).transpose(1, 2)
K = K.reshape(2, 10, 2, 2).transpose(1, 2)
V = V.reshape(2, 10, 2, 2).transpose(1, 2)

<a id="s05-scores"></a>
## Compute every query–key pair within each head

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Q [2,2,10,2] → Kᵀ [2,2,2,10] → Scores [2,2,10,10]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Q [2,2,10,2] → Kᵀ [2,2,2,10] → Scores [2,2,10,10]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Q [2,2,10,2]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">query rows</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-k)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-k)" font-size="24" text-anchor="middle" font-weight="650">Kᵀ [2,2,2,10]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">source columns</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Scores [2,2,10,10]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">two 10 × 10 grids</text></g></svg>

The first two axes keep examples and heads separate. Matrix multiplication contracts only the matching-coordinate axis.

In [13]:
scores = Q @ K.transpose(-2, -1)
scores = scores / math.sqrt(2)

<a id="s05-mask"></a>
## Block future sources in every head

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 287" role="img" aria-label="Receiving position; Allowed source positions" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Receiving position; Allowed source positions</title><g data-cell-left="20" data-cell-right="470"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Receiving position</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Allowed source positions</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1, 2</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1, 2, 3</text></g><path d="M20 216 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="470"><text x="30" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">10</text></g><g data-cell-left="470" data-cell-right="1140"><text x="480" y="254" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1 through 10</text></g><path d="M20 267 H1140" stroke="var(--line)"/></svg>

True marks a blocked entry. The same causal mask broadcasts across both examples and both heads.

In [14]:
future = torch.ones(10, 10, dtype=torch.bool).triu(1)
scores = scores.masked_fill(future, float("-inf"))

<a id="s05-softmax"></a>
## Each row becomes a distribution over sources

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 310" role="img" aria-label="Attention weights for head 1" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Attention weights for head 1</title><text x="15" y="33" fill="var(--c-q)" font-size="26" text-anchor="start" font-weight="500">Head 1 · final query at position 10</text><text x="165" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">The</text><rect x="120" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0838525915956979"/><text x="165" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.006</text><text x="266" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">fisherman</text><rect x="221" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.17962740297689267"/><text x="266" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.166</text><text x="367" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">sat</text><rect x="322" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08533354000056549"/><text x="367" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.009</text><text x="468" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">beside</text><rect x="423" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08738376966024583"/><text x="468" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.012</text><text x="569" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><rect x="524" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08453298482034709"/><text x="569" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.008</text><text x="670" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">river</text><rect x="625" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.5105865263773078"/><text x="670" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.718</text><text x="771" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">bank</text><rect x="726" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.11191242203024018"/><text x="771" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.053</text><text x="872" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">and</text><rect x="827" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="872" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="973" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">watched</text><rect x="928" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.0902221140911639"/><text x="973" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.017</text><text x="1074" y="85" fill="var(--ink)" font-size="18" text-anchor="middle" font-weight="500">the</text><rect x="1029" y="111" width="93" height="48" fill="var(--c-a)" fill-opacity="0.08327432422376953"/><text x="1074" y="145" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="500">0.005</text><text x="15" y="145" fill="var(--muted)" font-size="22" text-anchor="start" font-weight="500">weight</text><text x="580" y="270" fill="var(--ink)" font-size="26" text-anchor="middle" font-weight="500">One softmax over the ten sources. Row sum = 1.</text></svg>

The final axis indexes source tokens. Every allowed row sums to one; masked entries have weight zero.

In [15]:
A = scores.softmax(dim=-1)
assert A.shape == (2, 2, 10, 10)

In [16]:
expected = torch.tensor([[h['A'] for h in worksheet['headsLesson']['cases'][name]['heads']]
                         for name in ['river', 'cheque']])
torch.testing.assert_close(A, expected)
torch.testing.assert_close(A.sum(-1), torch.ones(2, 2, 10))
assert not A.triu(1).any()
print('All 400 head weights match the worksheet; no future source receives weight.')

All 400 head weights match the worksheet; no future source receives weight.


<a id="s05-mix"></a>
## Multiply the weights by the values

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="A [2,2,10,10] → V [2,2,10,2] → Messages [2,2,10,2]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>A [2,2,10,10] → V [2,2,10,2] → Messages [2,2,10,2]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">A [2,2,10,10]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">source weights</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">V [2,2,10,2]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">source content</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Messages [2,2,10,2]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one row per query/head</text></g></svg>

The source-token axis is summed out. Each head keeps its own two-coordinate message.

In [17]:
messages = A @ V

<a id="s05-join"></a>
## Put each token’s head messages side by side

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="[2,2,10,2] → [2,10,2,2] → [2,10,4]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>[2,2,10,2] → [2,10,2,2] → [2,10,4]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">[2,2,10,2]</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, heads, T, d_head</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">[2,10,2,2]</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, heads, d_head</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">[2,10,4]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">B, T, D</text></g></svg>

Transpose before reshaping. A direct reshape of the original layout would mix token rows with head rows.

In [18]:
joined = messages.transpose(1, 2).contiguous()
joined = joined.reshape(2, 10, 4)

<a id="s05-output-map"></a>
## Return to the shared prediction path

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="397.5" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="192" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q¹ · K¹ · V¹</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="412.5" y="312" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q² · K² · V²</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

The two head messages are joined. W_O mixes them, and the residual adds that update to the original E.

<a id="s05-output"></a>
## Project, add, and keep the final row

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Joined messages → W_O + residual → Last row → MLP" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Joined messages → W_O + residual → Last row → MLP</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Joined messages</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[2,10,4]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">updated E′ [2,10,4]</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">logits [2,20]</text></g></svg>

W_O combines the heads. The residual keeps E. Only the last updated row feeds this next-token loss.

In [19]:
delta = model.attention.W_O(joined)
updated = E + delta
logits = model.readout(F.relu(model.hidden(updated[:, -1])))

In [20]:
expected = torch.tensor([worksheet['headsLesson']['cases'][name]['logits'][-1]
                         for name in ['river', 'cheque']])
torch.testing.assert_close(logits, expected)
torch.testing.assert_close(model(X), logits)
print('All 40 vocabulary logits match the independent worksheet.')

All 40 vocabulary logits match the independent worksheet.


<a id="s06-break"></a>
## The rest of the learning loop is familiar

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

More heads change the context update, not the definition of the next-token target.

<a id="s06-loss"></a>
## Score the two observed targets

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Input X → Two heads + MLP → Logits [2,20] → Loss" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Input X → Two heads + MLP → Logits [2,20] → Loss</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Input X</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">2 × 10 token IDs</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Two heads + MLP</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one prediction/example</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Logits [2,20]</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">targets: water, teller</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Loss</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one scalar</text></g></svg>

Cross-entropy reads logits and observed token IDs. Do not sample a generated token to make the training target.

In [21]:
logits = model(X)
loss = F.cross_entropy(logits, y)

<a id="s06-update"></a>
## One loss trains all the projections

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Loss → Gradients → Optimizer → Updated weights" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Loss → Gradients → Optimizer → Updated weights</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-a)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-a)" font-size="24" text-anchor="middle" font-weight="650">Loss</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">from the same targets</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Gradients</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">all learned parameters</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Optimizer</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">one update</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Updated weights</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">used by the next batch</text></g></svg>

Heads are not assigned jobs or separate labels. They receive gradients from the same prediction loss.

In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [23]:
assert all(p.grad is not None and p.grad.isfinite().all() for p in model.parameters())
print('All learned tables, head projections and prediction layers received finite gradients.')

All learned tables, head projections and prediction layers received finite gradients.


<a id="s06-prompt"></a>
## Start generation from known token IDs

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="River prefix → Vocabulary lookup → History [1,10]" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>River prefix → Vocabulary lookup → History [1,10]</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">River prefix</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">the ten known words</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Vocabulary lookup</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">same vocabulary</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">History [1,10]</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">no observed next token</text></g></svg>

We already encoded the river sentence. A real application tokenizes and looks up a new prompt with the same vocabulary.

In [24]:
history = torch.tensor([river_ids])
model.eval()

TinyMultiHeadLM(
  (token_embedding): Embedding(20, 4)
  (position_embedding): Embedding(10, 4)
  (attention): ScratchMultiHead(
    (W_Q): Linear(in_features=4, out_features=4, bias=False)
    (W_K): Linear(in_features=4, out_features=4, bias=False)
    (W_V): Linear(in_features=4, out_features=4, bias=False)
    (W_O): Linear(in_features=4, out_features=4, bias=False)
  )
  (hidden): Linear(in_features=4, out_features=8, bias=True)
  (readout): Linear(in_features=8, out_features=20, bias=True)
)

<a id="s06-infer"></a>
## Generate one token, then repeat

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known IDs → Crop to context → Model logits → Choose + append" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known IDs → Crop to context → Model logits → Choose + append</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known IDs</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">no future target</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Crop to context</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">latest 10 in this toy</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Model logits</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">final row only</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Choose + append</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">new known prefix</text></g></svg>

At inference, weights stay fixed. The current prefix becomes longer; we keep only the configured context window.

In [25]:
with torch.no_grad():
    logits = model(history[:, -model.context:])
next_id = logits.argmax(dim=-1, keepdim=True)
history = torch.cat([history, next_id], dim=1)

<a id="s06-map"></a>
## Trace one request through the full model

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="120.0" y="50" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="397.5" y="50" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="192" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q¹ · K¹ · V¹</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="312" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q² · K² · V²</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="690.0" y="250" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

Start at the known IDs. Name the shape at each arrow. Where does the observed target enter during training? It enters only at the loss.

<a id="s07-break"></a>
## The same operation, packaged by PyTorch

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Which boxes does nn.MultiheadAttention replace?

<a id="s07-api-map"></a>
## Replace the head calculation, not the whole model

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 460" role="img" aria-label="Two heads inside the same next-token model" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Two heads inside the same next-token model</title><g><rect x="20" y="18" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="120.0" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Known tokens</text><text x="120.0" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">IDs [B,T]</text></g><g><rect x="280" y="18" width="235" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="397.5" y="50" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Lookup + position</text><text x="397.5" y="79" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E [B,T,D]</text></g><g><rect x="310" y="160" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="192" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 1</text><text x="412.5" y="221" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q¹ · K¹ · V¹</text></g><g><rect x="310" y="280" width="205" height="82" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="412.5" y="312" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Head 2</text><text x="412.5" y="341" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q² · K² · V²</text></g><g><rect x="590" y="218" width="200" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="690.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Concatenate</text><text x="690.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">messages [B,T,D]</text></g><g><rect x="900" y="218" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="250" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">W_O + residual</text><text x="1010.0" y="279" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">E′ = E + ΔE</text></g><g><rect x="900" y="365" width="220" height="82" rx="5" fill="white" stroke="var(--line)" stroke-width="2"/><text x="1010.0" y="397" fill="var(--muted)" font-size="24" text-anchor="middle" font-weight="650">Last row → MLP</text><text x="1010.0" y="426" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">next-token logits</text></g><path d="M220 59 L280 59 M270.78939005997114 55.1058165769135 L280 59 L270.78939005997114 62.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M397 100 L397 158 M400.89418342308653 148.78939005997114 L397 158 L393.10581657691347 148.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 100 L290 320 M293.89418342308653 310.78939005997114 L290 320 L286.10581657691347 310.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M290 320 L308 320 M298.78939005997114 316.10581657691347 L308 320 L298.78939005997114 323.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 201 L590 245 M584.0261332409137 236.9804666005631 L590 245 L580.0851058179404 243.69812698063134" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 321 L590 275 M580.1125370955748 276.4960204923785 L590 275 L584.184517668384 283.1351192523934" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M790 259 L900 259 M890.7893900599712 255.1058165769135 L900 259 L890.7893900599712 262.89418342308653" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M1010 300 L1010 365 M1013.8941834230865 355.78939005997114 L1010 365 L1006.1058165769135 355.78939005997114" fill="none" stroke="var(--muted)" stroke-width="2.5"/><path d="M515 59 H1010 V216" fill="none" stroke="var(--c-e)" stroke-width="2" stroke-dasharray="7 6"/><text x="680" y="45" fill="var(--c-e)" font-size="22" text-anchor="start" font-weight="500">Keep E for the residual</text></svg>

The PyTorch layer replaces both heads, their concatenation and W_O. We still add the residual and use our prediction MLP.

<a id="s07-api"></a>
## Create the multi-head attention layer

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Input E → nn.MultiheadAttention → Projected update ΔE" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Input E → nn.MultiheadAttention → Projected update ΔE</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Input E</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[B,T,4]</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">nn.MultiheadAttention</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">2 heads × 2 coordinates</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Projected update ΔE</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">[B,T,4]</text></g></svg>

batch_first=True means [B,T,D]. bias=False and dropout=0 match our scratch implementation.

In [26]:
mha = nn.MultiheadAttention(
    embed_dim=4, num_heads=2, bias=False,
    dropout=0.0, batch_first=True)

<a id="s07-call"></a>
## Pass E as query, key and value input

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="E, E, E → nn.MultiheadAttention → ΔE and A" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>E, E, E → nn.MultiheadAttention → ΔE and A</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">E, E, E</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">unprojected input rows</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">nn.MultiheadAttention</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">Q/K/V, softmax, mix, W_O</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">ΔE and A</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">not a residual yet</text></g></svg>

PyTorch performs the learned projections internally. average_attn_weights=False preserves the head axis in the returned weight tensor.

In [27]:
delta, A = mha(E, E, E, attn_mask=future,
               average_attn_weights=False)
updated = E + delta

<a id="s07-check"></a>
## Compare the same weights, not two random layers

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Quantity; Expected shape" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Quantity; Expected shape</title><g data-cell-left="20" data-cell-right="580"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Quantity</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Expected shape</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Projected update ΔE</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2,10,4]</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Separate head weights A</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2,2,10,10]</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="580"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Residual E + ΔE</text></g><g data-cell-left="580" data-cell-right="1140"><text x="590" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">[2,10,4]</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

The notebook copies the scratch Q/K/V and W_O parameters into PyTorch, then checks both outputs numerically.

In [28]:
mha = copy_to_pytorch(model.attention)
delta_scratch, A_scratch = model.attention(E)
delta_api, A_api = mha(E, E, E, attn_mask=future,
                      average_attn_weights=False)
torch.testing.assert_close(delta_api, delta_scratch)

In [29]:
torch.testing.assert_close(A_api, A_scratch)
print('Scratch and PyTorch updates agree:', tuple(delta_api.shape))
print('Individual head weights agree:', tuple(A_api.shape))

Scratch and PyTorch updates agree: (2, 10, 4)
Individual head weights agree: (2, 2, 10, 10)


<a id="s07-boundary"></a>
## What the attention block does not include

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Token + position → Multi-head attention → Residual → Prediction MLP" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Token + position → Multi-head attention → Residual → Prediction MLP</title><g><rect x="20.0" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="146.875" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Token + position</text><text x="146.875" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">outside the API</text></g><path d="M273.75 113 L306.75 113 M297.53939005997114 109.1058165769135 L306.75 113 L297.53939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="308.75" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="435.625" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Multi-head attention</text><text x="435.625" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">returns projected ΔE</text></g><path d="M562.5 113 L595.5 113 M586.2893900599712 109.1058165769135 L595.5 113 L586.2893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="597.5" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="724.375" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">Residual</text><text x="724.375" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">add E ourselves</text></g><path d="M851.25 113 L884.25 113 M875.0393900599712 109.1058165769135 L884.25 113 L875.0393900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="886.25" y="70" width="253.75" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="1013.125" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Prediction MLP</text><text x="1013.125" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">outside the API</text></g></svg>

nn.MultiheadAttention does not add position embeddings, the residual, a vocabulary head or a training loop.

<a id="s08-break"></a>
## Do more heads help this experiment?

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Known inputs → Two heads → One prediction" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Known inputs → Two heads → One prediction</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Known inputs</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Two heads</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-d)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-d)" font-size="24" text-anchor="middle" font-weight="650">One prediction</text></g></svg>

Move from the small worksheet to the actual TinyStories checkpoints.

<a id="s08-width"></a>
## More heads need not mean more parameters

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 185" role="img" aria-label="Setting; Total width; Heads; Width per head" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Setting; Total width; Heads; Width per head</title><g data-cell-left="20" data-cell-right="440"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Setting</text></g><g data-cell-left="440" data-cell-right="670"><text x="450" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Total width</text></g><g data-cell-left="670" data-cell-right="870"><text x="680" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Heads</text></g><g data-cell-left="870" data-cell-right="1140"><text x="880" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Width per head</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">One head</text></g><g data-cell-left="440" data-cell-right="670"><text x="450" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">64</text></g><g data-cell-left="670" data-cell-right="870"><text x="680" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1</text></g><g data-cell-left="870" data-cell-right="1140"><text x="880" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">64</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Four heads</text></g><g data-cell-left="440" data-cell-right="670"><text x="450" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">64</text></g><g data-cell-left="670" data-cell-right="870"><text x="680" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">4</text></g><g data-cell-left="870" data-cell-right="1140"><text x="880" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">16</text></g><path d="M20 165 H1140" stroke="var(--line)"/></svg>

Q, K, V and W_O remain 64×64. The number of attention patterns grows, while each pattern uses fewer matching coordinates.

<a id="s08-scores"></a>
## Four heads improved held-out prediction here

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Model; Test cross-entropy ↓; Test perplexity ↓" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Model; Test cross-entropy ↓; Test perplexity ↓</title><g data-cell-left="20" data-cell-right="430"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Model</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Test cross-entropy ↓</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Test perplexity ↓</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="430"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">MLP</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.943</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">51.59</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="430"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">One head</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.445</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">31.34</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="430"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Four heads</text></g><g data-cell-left="430" data-cell-right="785"><text x="440" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">3.360</text></g><g data-cell-left="785" data-cell-right="1140"><text x="795" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">28.78</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

Three-seed means. Same stories, tokenizer, context and update budget; validation-selected checkpoints. Perplexity = exp(cross-entropy). Lower is better. This does not guarantee better text for every prompt.

<a id="s08-costs"></a>
## Compare the cost as well as the score

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Model; Parameters; Training per seed" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Model; Parameters; Training per seed</title><g data-cell-left="20" data-cell-right="420"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Model</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Parameters</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Training per seed</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">MLP</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">2,332,832</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">58.2 s</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">One head</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1,321,120</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">65.6 s</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="420"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Four heads</text></g><g data-cell-left="420" data-cell-right="770"><text x="430" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">1,321,120</text></g><g data-cell-left="770" data-cell-right="1140"><text x="780" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">69.7 s</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

Mean training time on Apple M2 Max/MPS, including validation checks. Both attention models have equal parameter counts; the MLP is larger.

<a id="s08-demo"></a>
## Try the same prompt with all three models

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="Training prefix → Held-out prefix → Outside-domain prompt" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Training prefix → Held-out prefix → Outside-domain prompt</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Training prefix</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">familiar text</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Held-out prefix</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">new story, same task</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">Outside-domain prompt</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">different kind of text</text></g></svg>

<a href="../../word-lab/" target="_blank" rel="noopener">Open the live browser demo ↗</a> Compare continuations, vocabulary coverage and generation time. This runs real checkpoints with WebGPU or WASM.

<a id="s08-notebooks"></a>
## Run the calculation yourself

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 236" role="img" aria-label="Resource; What to do" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>Resource; What to do</title><g data-cell-left="20" data-cell-right="440"><text x="30" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">Resource</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="50" fill="var(--muted)" font-size="23" text-anchor="start" font-weight="650">What to do</text></g><path d="M20 63 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Notebook 7</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="101" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Follow every operation and check PyTorch parity</text></g><path d="M20 114 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Notebook 6</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="152" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Inspect the trained four-head checkpoint and results</text></g><path d="M20 165 H1140" stroke="var(--line)"/><g data-cell-left="20" data-cell-right="440"><text x="30" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Part 2B · optional</text></g><g data-cell-left="440" data-cell-right="1140"><text x="450" y="203" fill="var(--ink)" font-size="26" text-anchor="start" font-weight="500">Gradients, normalization, full blocks and context cost</text></g><path d="M20 216 H1140" stroke="var(--line)"/></svg>

<a href="../../notebooks/wordlm/07_multihead_step_by_step.html">Step-by-step notebook</a> · <a href="../../notebooks/wordlm/06_multihead_comparison.html">Measured experiment</a> · <a href="../../part2b.html">Optional reference</a>

<a id="s08-next"></a>
## Next: read a different sequence

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 225" role="img" aria-label="French decoder row → Query → English keys + values" style="font-family:Avenir Next,Segoe UI,sans-serif;--c-e:#245EDB;--c-q:#8B2CDE;--c-k:#AA4E08;--c-v:#0F766E;--c-a:#BE123C;--c-d:#147737;--ink:#14171F;--muted:#50586a;--line:#D9DFE9"><title>French decoder row → Query → English keys + values</title><g><rect x="20.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-e)" stroke-width="2"/><text x="195.0" y="102" fill="var(--c-e)" font-size="24" text-anchor="middle" font-weight="650">French decoder row</text><text x="195.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">the next output token</text></g><path d="M370.0 113 L403.0 113 M393.78939005997114 109.1058165769135 L403.0 113 L393.78939005997114 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="405.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-q)" stroke-width="2"/><text x="580.0" y="102" fill="var(--c-q)" font-size="24" text-anchor="middle" font-weight="650">Query</text><text x="580.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">what do I need?</text></g><path d="M755.0 113 L788.0 113 M778.7893900599712 109.1058165769135 L788.0 113 L778.7893900599712 116.8941834230865" fill="none" stroke="var(--muted)" stroke-width="2.5"/><g><rect x="790.0" y="70" width="350.0" height="86" rx="5" fill="white" stroke="var(--c-v)" stroke-width="2"/><text x="965.0" y="102" fill="var(--c-v)" font-size="24" text-anchor="middle" font-weight="650">English keys + values</text><text x="965.0" y="131" fill="var(--muted)" font-size="21" text-anchor="middle" font-weight="500">where should I read?</text></g></svg>

So far, Q, K and V came from the same sequence. <a href="../../part4.html">Part IV changes that: cross-attention reads another sequence.</a>

## Continue with the trained experiment

[Notebook 6](06_multihead_comparison.html) inspects the four-head TinyStories
checkpoint, per-head weights and all three-seed results. The browser demo loads
the actual exported checkpoints and measures generation on your device.

Visual teaching references: [3Blue1Brown](https://www.3blue1brown.com/lessons/attention/)
and [Jay Alammar](https://jalammar.github.io/illustrated-transformer/).
Formula and API references: [Attention Is All You Need, §3.2.2](https://arxiv.org/abs/1706.03762)
and [PyTorch MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html).
The diagrams and worked numbers here are original adaptations of our Part II example.